# Saxo OpenAPI — Instrument Resolver & Cache (avec gestion Rate Limit)

Ce notebook :
- Authentifie via OAuth2 PKCE (Saxo OpenAPI)
- Résout une fois les instruments actions (ticker → UIC)
- Stocke les résultats dans un cache persistant (Parquet)
- Liste et exporte les *unresolved* pour correction
- Inclut une gestion robuste du *rate limit exceeded* (HTTP 429)


In [ ]:
!pip -q install pandas pyarrow requests

import os
import time
import json
import base64
import hashlib
import secrets
import urllib.parse
from dataclasses import dataclass
import pandas as pd
import requests
import re


## Configuration (Google Drive + variables d'environnement)
Recommandé : stocker le cache dans Google Drive pour persistance entre sessions Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Chemin persistant du cache instruments
os.environ.setdefault('BASE_CACHE_DIR', "/content/drive/MyDrive/saxo/cache")

# Dossier Drive où tu stockes tes univers
os.environ.setdefault('UNIVERSE_DIR', "/content/drive/MyDrive/saxo/universes")

# Dossier Drive où tu stockes tes univers
os.environ.setdefault('BASE_TRADE_DIR', "/content/drive/MyDrive/saxo/trades")


# Environnement SIM (modifiable vers LIVE plus tard)
os.environ.setdefault('SAXO_OPENAPI_BASE', 'https://gateway.saxobank.com/sim/openapi')
os.environ.setdefault('SAXO_AUTH_BASE', 'https://sim.logonvalidation.net')

# À définir via Colab Secrets ou ici temporairement
# os.environ['SAXO_CLIENT_ID'] = 'TON_CLIENT_ID'
# os.environ['SAXO_REDIRECT_URI'] = 'TON_REDIRECT_URI'

markets_cfg = [
    {
        "market": "us",
        "label": "United States",
        "universe_name": "us_IWB_Russell1000",
        "ishares_url": "https://www.ishares.com/us/products/239707/ishares-russell-1000-etf/1467271812596.ajax?fileType=csv&fileName=IWB_holdings&dataType=fund",
        "benchmark_symbol": "IWB",   # proxy Russell 1000
        "update_mode": "add",       # "add" ou "recreate"

    },
    {
        "market": "eu",
        "label": "Europe",
        "universe_name": "eu_IWB_STOXX600",
        "ishares_url": "https://www.ishares.com/ch/privatkunden/de/produkte/251931/ishares-stoxx-europe-600-ucits-etf-de-fund/1495092304805.ajax?fileType=csv&fileName=EXSA_holdings&dataType=fund",
        "benchmark_symbol": "EXSA",  # STOXX 600 UCITS
        "update_mode": "add",       # "add" ou "recreate"

    },
]


# Mapping NOM DE PLACE (iShares) -> code marché Saxo/MIC-like
EXCHANGE_NAME_TO_SAXO = {
    # --- US ---
    "NASDAQ": "XNAS",
    "NASDAQ STOCK MARKET": "XNAS",
    "NASDAQ OMX": "XNAS",
    "NEW YORK STOCK EXCHANGE": "XNYS",
    "NEW YORK STOCK EXCHANGE INC.": "XNYS",
    "NYSE": "XNYS",
    "NYSE MKT LLC": "XASE",
    "NYSE AMERICAN": "XASE",

    # --- Europe ---
    "EURONEXT PARIS": "XPAR",
    "NYSE EURONEXT - EURONEXT PARIS": "XPAR",
    "EURONEXT AMSTERDAM": "XAMS",
    "NYSE EURONEXT - EURONEXT AMSTERDAM": "XAMS",
    "EURONEXT BRUSSELS": "XBRU",
    "NYSE EURONEXT - EURONEXT BRUSSELS": "XBRU",
    "EURONEXT LISBON": "XLIS",
    "NYSE EURONEXT - EURONEXT LISBON": "XLIS",

    "LONDON STOCK EXCHANGE": "XLON",
    "IRISH STOCK EXCHANGE": "XDUB",

    "XETRA": "XETR",
    "BÖRSE FRANKFURT": "XFRA",
    "BORSE FRANKFURT": "XFRA",

    "SIX SWISS EXCHANGE": "XSWX",
    "BOLSA DE MADRID": "XMAD",
    "BORSA ITALIANA": "XMIL",
    "WIENER BOERSE": "XWBO",
    "WARSAW STOCK EXCHANGE": "XWAR",
    "OSLO BORS": "XOSL",

    "NASDAQ OMX HELSINKI": "XHEL",
    "NASDAQ OMX NORDIC": "XNSE",  # générique nordique
    "OMX NORDIC EXCHANGE COPENHAGEN": "XCSE",
    "OMX NORDIC EXCHANGE COPENHAGEN A/S": "XCSE",
}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## OAuth2 PKCE helpers

In [ ]:
def pkce_verifier_challenge():
    verifier = secrets.token_urlsafe(64)[:128]
    challenge = base64.urlsafe_b64encode(
        hashlib.sha256(verifier.encode('utf-8')).digest()
    ).decode('utf-8').rstrip('=')
    return verifier, challenge

def build_auth_url(client_id: str, redirect_uri: str, auth_base: str):
    verifier, challenge = pkce_verifier_challenge()
    state = secrets.token_urlsafe(16)
    params = {
        'response_type': 'code',
        'client_id': client_id,
        'redirect_uri': redirect_uri,
        'code_challenge': challenge,
        'code_challenge_method': 'S256',
        'state': state,
    }
    url = f"{auth_base}/authorize?{urllib.parse.urlencode(params)}"
    return url, verifier

def exchange_code_for_token(code: str, client_id: str, redirect_uri: str, verifier: str, auth_base: str):
    token_url = f"{auth_base}/token"
    data = {
        'grant_type': 'authorization_code',
        'code': code,
        'redirect_uri': redirect_uri,
        'client_id': client_id,
        'code_verifier': verifier,
    }
    r = requests.post(token_url, data=data, timeout=30)
    r.raise_for_status()
    return r.json()


## Auth interactive (copier/coller le `code`)

In [ ]:
from google.colab import userdata
import secrets

auth_base = "https://sim.logonvalidation.net"
client_id = userdata.get('SAXO_APP_KEY').strip()
redirect_uri = userdata.get('SAXO_REDIRECT_URI').strip()

auth_url, verifier = build_auth_url(client_id, redirect_uri, auth_base)
print('Ouvre cette URL dans un navigateur, connecte-toi, accepte :')
print(auth_url)

Ouvre cette URL dans un navigateur, connecte-toi, accepte :
https://sim.logonvalidation.net/authorize?response_type=code&client_id=d136951a48744e61a28ddec3ddf9327c&redirect_uri=https%3A%2F%2Fgithub.com%2Ftixounet%2Fsaxo-redirect%2F&code_challenge=7yfL63KV4RS573MbSmZesE8QPFc4bELxAjNl1ByFEqA&code_challenge_method=S256&state=ZMeFvIQAlJGuuv4jSZXV4w


In [ ]:
from urllib.parse import urlparse, parse_qs;

# Colle le paramètre `code` obtenu dans l'URL de redirection
url = 'https://github.com/tixounet/saxo-redirect/?code=87f02361-4d65-41c7-ad7f-03763bfc4a93&state=ZMeFvIQAlJGuuv4jSZXV4w#/lst/1768983799464'


code_param = parse_qs(urlparse(url).query)["code"][0]

token = exchange_code_for_token(code_param, client_id, redirect_uri, verifier, auth_base)
access_token = token['access_token']
headers = {'Authorization': f'Bearer {access_token}'}
print('Token OK. Extrait:', access_token[:20], '...')


Token OK. Extrait: eyJhbGciOiJFUzI1NiIs ...


## Client REST Saxo avec gestion du rate limit (HTTP 429)
Gestion :
- si HTTP 429 : utilise `Retry-After` si présent, sinon backoff exponentiel
- gère aussi certains 5xx transitoires
- détecte aussi le message 'rate limit exceeded' dans le payload quand applicable

In [ ]:
OPENAPI_BASE = os.environ['SAXO_OPENAPI_BASE']

class SaxoRateLimitError(Exception):
    pass

def _get_retry_after_seconds(resp: requests.Response) -> float | None:
    ra = resp.headers.get('Retry-After')
    if not ra:
        return None
    try:
        return float(ra)
    except ValueError:
        return None

def saxo_get(path: str, params: dict | None = None, max_retries: int = 8, base_sleep: float = 1.0):
    url = f"{OPENAPI_BASE}{path}"
    last_err = None
    sleep_min = 60
    for attempt in range(1, max_retries + 1):
        resp = requests.get(url, headers=headers, params=params, timeout=30)

        # Rate limit
        if resp.status_code == 429:
            retry_after = _get_retry_after_seconds(resp)
            sleep_s = retry_after if retry_after is not None else min(60.0, sleep_min + base_sleep * (2 ** (attempt - 1)))
            # Try to parse message for logging
            msg = None
            try:
                j = resp.json()
                msg = j.get('Message') or j.get('message') or str(j)[:200]
            except Exception:
                msg = resp.text[:200]
            print(f"[429 Rate limit] attempt {attempt}/{max_retries} — sleeping {sleep_s:.1f}s — {msg}")
            time.sleep(sleep_s)
            last_err = SaxoRateLimitError(msg)
            continue

        # Transient gateway/server errors
        if resp.status_code in (502, 503, 504):
            sleep_s = min(30.0, base_sleep * (2 ** (attempt - 1)))
            print(f"[{resp.status_code}] attempt {attempt}/{max_retries} — sleeping {sleep_s:.1f}s")
            time.sleep(sleep_s)
            last_err = RuntimeError(resp.text[:200])
            continue

        # Other errors
        if resp.status_code >= 400:
            # Some APIs return rate limit message with non-429; detect it
            body = resp.text or ''
            if 'rate limit' in body.lower() and 'exceed' in body.lower():
                sleep_s = min(60.0, base_sleep * (2 ** (attempt - 1)))
                print(f"[Rate limit message] status {resp.status_code} attempt {attempt}/{max_retries} — sleeping {sleep_s:.1f}s")
                time.sleep(sleep_s)
                last_err = SaxoRateLimitError(body[:200])
                continue
            else:
                raise RuntimeError(f"{resp.status_code} {resp.reason} for url: {resp.url}\n{resp.text}")
            resp.raise_for_status()

        # Success
        return resp.json()

    if last_err:
        raise last_err
    raise RuntimeError('Request failed after retries')


## Instruments search + résolution (ticker → UIC)

**Récupérer les Tickers**
Télécharger le holdings iShares et le sauvegarder sur Drive

In [ ]:
import os
import re
import pandas as pd
import requests
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

def _with_filetype(url: str, filetype: str) -> str:
    """
    Reprend une URL iShares du type:
      .../?dataType=fund&fileName=IWB_holdings&fileType=xls
    et force fileType=csv (ou xls).
    """
    u = urlparse(url)
    q = parse_qs(u.query)
    q["fileType"] = [filetype]  # IMPORTANT: casse exacte des paramètres iShares  [oai_citation:1‡Stack Overflow](https://stackoverflow.com/questions/72210552/unable-to-download-csv-file-from-ishares-website?utm_source=chatgpt.com)
    new_query = urlencode({k: v[0] for k, v in q.items()})
    return urlunparse((u.scheme, u.netloc, u.path, u.params, new_query, u.fragment))

def download_ishares_holdings(product_url: str, out_path: str, prefer: str = "csv") -> str:
    """
    Télécharge le fichier holdings iShares vers out_path.
    - prefer="csv" essaie d'abord fileType=csv puis fallback en xls.
    Retourne le chemin réellement écrit.
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0"
    })

    tried = []
    for ft in ([prefer, "xls"] if prefer == "csv" else [prefer, "csv"]):
        dl_url = _with_filetype(product_url, ft)
        tried.append(dl_url)
        r = session.get(dl_url, timeout=60)
        if r.ok and len(r.content) > 1000:
            with open(out_path, "wb") as f:
                f.write(r.content)
            return out_path

    raise RuntimeError(f"Echec téléchargement iShares. URLs testées: {tried}")

def read_holdings_file(path: str) -> pd.DataFrame:
    """
    Lit un fichier holdings iShares (csv ou xls) en DataFrame.
    Gère les entêtes parfois précédées de lignes de disclaimer.
    """
    ext = os.path.splitext(path)[1].lower()

    if ext == ".csv":
        # Certains CSV iShares ont des lignes avant le header.
        # On détecte la première ligne contenant "Ticker" (ou "ISIN") et on la prend comme header.
        with open(path, "rb") as f:
            raw = f.read().decode("utf-8", errors="replace").splitlines()

        header_idx = None
        for i, line in enumerate(raw[:80]):
            if re.search(r"\bTicker\b", line) or re.search(r"\bEmittententicker\b", line) or re.search(r"\bISIN\b", line):
                header_idx = i
                break
        if header_idx is None:
            header_idx = 0

        from io import StringIO
        csv_text = "\n".join(raw[header_idx:])
        df = pd.read_csv(StringIO(csv_text))
        return df

    # XLS / XLSX
    # iShares fournit souvent du .xls (lisible par pandas via engine auto)
    df = pd.read_excel(path)
    return df

def extract_tickers_from_holdings(df: pd.DataFrame) -> pd.DataFrame:
    """
    Retourne un DataFrame avec:
      - symbol         (ticker)
      - name           (nom / description iShares si dispo)
      - exchange_name  (nom de place: "NASDAQ", "London Stock Exchange", "Xetra", ... si dispo)
      - market_currency (devise de cotation: "USD", "EUR", ... si dispo)

    Notes:
    - exchange_name est un NOM (pas un code Saxo). On le conservera pour un mapping ultérieur vers ExchangeId Saxo.
    - market_currency servira aussi au scoring de match côté Saxo.
    """
    cols = {c.lower().strip(): c for c in df.columns}

    # --- Ticker column detection ---
    ticker_col = None
    for candidate in ["ticker", "emittententicker", "issuer ticker", "issue ticker"]:
        if candidate in cols:
            ticker_col = cols[candidate]
            break
    if ticker_col is None:
        return pd.DataFrame(columns=["symbol", "name", "exchange_name", "market_currency"])

    # --- Name/Description column detection ---
    name_col = None
    for candidate in [
        "name", "security name", "security", "description", "issuer name",
        "holding name", "instrument name", "wertpapierbezeichnung",
        "bezeichnung", "titel", "nom",
    ]:
        if candidate in cols:
            name_col = cols[candidate]
            break

    # --- Exchange column detection (EN/DE) ---
    exchange_col = None
    for candidate in [
        "exchange", "börse", "borse", "stock exchange", "listing exchange",
        "börse/marktplatz", "handelsplatz"
    ]:
        if candidate in cols:
            exchange_col = cols[candidate]
            break

    # --- Market Currency column detection (EN/DE) ---
    ccy_col = None
    for candidate in [
        "market currency", "marktwährung", "marktwahrung", "currency", "währung", "waehrung"
    ]:
        if candidate in cols:
            ccy_col = cols[candidate]
            break

    work = df.copy()

    # --- Optional filtering if columns exist ---
    if "asset class" in cols:
        ac = cols["asset class"]
        work = work[work[ac].astype(str).str.lower().str.contains("equity", na=False)]
    elif "anlageklasse" in cols:
        ac = cols["anlageklasse"]
        work = work[work[ac].astype(str).str.lower().str.contains("aktien", na=False)]
    elif "security type" in cols:
        st = cols["security type"]
        work = work[work[st].astype(str).str.lower().str.contains("common|equity|stock|sh", na=False)]

    out = pd.DataFrame()

    out["symbol"] = (
        work[ticker_col]
        .dropna()
        .astype(str)
        .str.strip()
        .str.upper()
    )

    out["name"] = work[name_col].astype(str).str.strip() if name_col is not None else None
    out["exchange_name"] = work[exchange_col].astype(str).str.strip() if exchange_col is not None else None
    out["market_currency"] = work[ccy_col].astype(str).str.strip().str.upper() if ccy_col is not None else None

    # --- Filter invalid tickers ---
    out = out[out["symbol"].apply(lambda t: bool(re.fullmatch(r"[A-Z0-9\.\-]{1,15}", t)) and t != "-")]

    # --- Clean placeholders ---
    for c in ["name", "exchange_name", "market_currency"]:
        if c in out.columns:
            out[c] = out[c].replace({"nan": None, "None": None, "": None})

    # --- Dedup: keep first row encountered per symbol ---
    out = out.drop_duplicates(subset=["symbol"], keep="first").reset_index(drop=True)

    return out

def rebuild_ticker_map_from_holdings_file(path: str) -> pd.DataFrame:
    """
    Recharge un fichier holdings iShares déjà téléchargé (csv ou xls)
    et reconstruit un DataFrame propre avec:
      - symbol (ticker)
      - name   (nom / description)

    Aucun téléchargement n'est effectué.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Holdings file not found: {path}")

    # 1) Read raw holdings file (csv/xls, robust to disclaimers)
    df_raw = read_holdings_file(path)

    if df_raw is None or len(df_raw) == 0:
        raise RuntimeError(f"Holdings file is empty or unreadable: {path}")

    # 2) Extract (symbol, name) using the hardened extractor
    df_tickers = extract_tickers_from_holdings(df_raw)

    if len(df_tickers) == 0:
        raise RuntimeError(f"No valid tickers extracted from holdings file: {path}")

    return df_tickers

**Télécharger les indices depuis internet et les sauvegarder dans Universe**

A n'exécuter que si un nouveau marché ou un refresh

In [ ]:
# Dossier Drive où tu stockes tes univers
UNIVERSE_DIR = os.environ['UNIVERSE_DIR']
os.makedirs(UNIVERSE_DIR, exist_ok=True)

def build_universe_from_ishares(product_url: str, universe_name: str) -> str:
    """
    Télécharge le holdings iShares, extrait les tickers, et sauvegarde un CSV "universe".
    Retourne le chemin du CSV créé.
    """
    holdings_path = os.path.join(UNIVERSE_DIR, f"{universe_name}_holdings.csv")
    universe_csv  = os.path.join(UNIVERSE_DIR, f"{universe_name}_tickers.csv")

    # 1) download holdings (on sauvegarde en .csv côté disque même si fallback xls est possible)
    download_ishares_holdings(product_url, holdings_path, prefer="csv")

    # 2) read + extract tickers
    df_hold = read_holdings_file(holdings_path)
    tickers = extract_tickers_from_holdings(df_hold)

    # 3) save tickers file
    tickers.to_csv(universe_csv, index=False)

    print(f"Universe '{universe_name}' créé: {universe_csv} ({len(tickers)} tickers)")
    return universe_csv

for cfg in markets_cfg:
    print(f"▶ Build universe for {cfg['market']} ({cfg['label']})")
    build_universe_from_ishares(
        cfg["ishares_url"],
        universe_name=cfg["universe_name"]
    )


▶ Build universe for us (United States)


ValueError: If using all scalar values, you must pass an index

In [ ]:
# !!!!!!!!!!!!!!
# A executer UNIQUEMENT pour recréer le ticker file sur base du holding file sans télécharger
# !!!!!!!!!!!!!!

UNIVERSE_DIR = os.environ['UNIVERSE_DIR']

ticker_map_by_market = {}

for cfg in markets_cfg:
    m = cfg["market"]
    universe = cfg["universe_name"]

    print(f"▶ Reload universe from holdings for {m} ({cfg['label']})")

    # Chemin attendu du fichier holdings
    holdings_path_csv = os.path.join(
        UNIVERSE_DIR, f"{universe}_holdings.csv"
    )
    holdings_path_xls = os.path.join(
        UNIVERSE_DIR, f"{universe}_holdings.xls"
    )

    if os.path.exists(holdings_path_csv):
        holdings_path = holdings_path_csv
    elif os.path.exists(holdings_path_xls):
        holdings_path = holdings_path_xls
    else:
        raise FileNotFoundError(
            f"[{m}] Aucun fichier holdings trouvé pour {universe}"
        )

    df_ticker_map = rebuild_ticker_map_from_holdings_file(holdings_path)

    # --- Save tickers file (persistent, reusable) ---
    out_tickers_path = os.path.join(
        UNIVERSE_DIR, f"{universe}_tickers.csv"
    )
    df_ticker_map.to_csv(out_tickers_path, index=False)

    ticker_map_by_market[m] = df_ticker_map

    print(
        f"✔ [{m}] Reload OK: "
        f"{len(df_ticker_map)} tickers "
        f"(ex: {df_ticker_map.iloc[0]['symbol']} – {df_ticker_map.iloc[0]['name']})"
    )

▶ Reload universe from holdings for eu (Europe)
✔ [eu] Reload OK: 551 tickers (ex: ASML – ASML HOLDING NV)
▶ Reload universe from holdings for us (United States)
✔ [us] Reload OK: 1008 tickers (ex: NVDA – NVIDIA CORP)


## Cache persistant (Parquet)
**Si les indices sont déjà sur drive, charger les fichiers universe**

**Vérifier si les actions existent sur Saxo**
et sauvegarder dans un fichier cache les actions connues

créér un fichier avec les inconnus à corriger


In [ ]:
import os
import pandas as pd
import glob
import re
from difflib import SequenceMatcher


UNIVERSE_DIR = os.environ['UNIVERSE_DIR']

def list_universe_files() -> list[str]:
    return sorted(glob.glob(os.path.join(UNIVERSE_DIR, "*_tickers.csv")))

def load_universe(universe_csv_path: str) -> pd.DataFrame:
    """
    Charge un fichier universe *_tickers.csv et retourne le DataFrame COMPLET.

    Exigences minimales:
      - colonne 'symbol'

    Colonnes supplémentaires conservées telles quelles:
      - name
      - exchange_name
      - market_currency
      - etc.
    """
    if not os.path.exists(universe_csv_path):
        raise FileNotFoundError(f"Universe file not found: {universe_csv_path}")

    df_uni = pd.read_csv(universe_csv_path)

    if "symbol" not in df_uni.columns:
        raise ValueError("Le fichier univers doit contenir une colonne 'symbol'")

    # Normalisation minimale (sécurité)
    df_uni["symbol"] = (
        df_uni["symbol"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Drop lignes sans symbole
    df_uni = df_uni.dropna(subset=["symbol"]).reset_index(drop=True)

    return df_uni
# ---------- Cache I/O (multi-caches by market) ----------

def cache_path_for_market(base_dir: str, market: str) -> str:
    market = market.strip().lower()
    os.makedirs(base_dir, exist_ok=True)
    return os.path.join(base_dir, f"instruments_{market}.parquet")

def load_instrument_cache_market(base_dir: str, market: str) -> pd.DataFrame | None:
    path = cache_path_for_market(base_dir, market)
    if os.path.exists(path):
        return pd.read_parquet(path)
    return None

def save_instrument_cache_market(df: pd.DataFrame, base_dir: str, market: str) -> str:
    path = cache_path_for_market(base_dir, market)
    df.to_parquet(path, index=False)
    return path

def combine_market_caches(base_dir: str, markets: list[str]) -> pd.DataFrame:
    dfs = []
    for m in markets:
        df = load_instrument_cache_market(base_dir, m)
        if df is not None and len(df) > 0:
            df = df.copy()
            df["market"] = m
            dfs.append(df)
    if not dfs:
        return pd.DataFrame(columns=["symbol","status","reason","uic","asset_type","exchange_id","currency","description","primary_listing","market"])
    out = pd.concat(dfs, ignore_index=True)
    # Dedup strict on (market, symbol) keeping latest row
    out = out.drop_duplicates(subset=["market","symbol"], keep="last")
    return out

# ---------- Helpers ----------

def normalize_tickers(tickers: list[str]) -> list[str]:
    return sorted({t.strip().upper() for t in tickers if isinstance(t, str) and t.strip()})

def _norm_txt(s: str) -> str:
    s = (s or "").upper()
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def name_similarity(a: str, b: str) -> float:
    a2, b2 = _norm_txt(a), _norm_txt(b)
    if not a2 or not b2:
        return 0.0
    return SequenceMatcher(None, a2, b2).ratio()

def normalize_exchange_name(name: str | None) -> str | None:
    if not name:
        return None
    s = name.upper()
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def enrich_universe_with_saxo_market(df_uni: pd.DataFrame) -> pd.DataFrame:
    """
    Ajoute/refresh la colonne 'saxo_market' (ex: XNYS, XETR, XPAR) à partir de exchange_name.
    Ne crée pas de colonnes dupliquées.
    """
    df = df_uni.copy()
    if "saxo_market" not in df.columns:
        df["saxo_market"] = None

    if "exchange_name" not in df.columns:
        # pas d'info exchange => on ne peut pas mapper, on laisse tel quel
        return df

    df["saxo_market"] = df["exchange_name"].apply(map_exchange_name_to_saxo)
    return df

# ------------- Market Resolve --------------

def normalize_exchange_name(name: str | None) -> str | None:
    if not name:
        return None
    s = name.upper()
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def map_exchange_name_to_saxo(exchange_name: str | None) -> str | None:
    """
    Traduit un nom de place (iShares) vers un code marché Saxo/MIC-like.
    Retourne None si non reconnu.
    """
    norm = normalize_exchange_name(exchange_name)
    if not norm:
        return None

    # exact match
    if norm in EXCHANGE_NAME_TO_SAXO:
        return EXCHANGE_NAME_TO_SAXO[norm]

    # fallback par inclusion (robuste)
    for k, v in EXCHANGE_NAME_TO_SAXO.items():
        if k in norm:
            return v

    return None

# ---------- Resolve (your existing functions reused) ----------

def instrument_search(keyword: str, asset_types: str = "Stock", top: int = 25, exchange_id: str | None = None):
    params = {"$top": top, "Keywords": keyword, "AssetTypes": asset_types}
    if exchange_id:
        params["ExchangeId"] = exchange_id
    return saxo_get("/ref/v1/instruments", params=params)


def pick_best_stock_match(
    keyword: str,
    data: list[dict],
    preferred_mics: list[str] | None = None,      # ex: ["XNAS","XNYS","XETR"]
    expected_currency: str | None = None,         # ex: "USD"
    expected_name: str | None = None,             # iShares name
) -> dict | None:
    """
    Saxo instruments often return Symbol like "NVDA:XNAS" (MIC in Symbol suffix),
    while ExchangeId can be a venue name like "NASDAQ".

    This picker:
      - compares keyword against the root symbol (before ':')
      - boosts candidates whose MIC suffix matches preferred_mics
      - keeps other heuristics (currency, primary listing, name similarity)
    """
    if not data:
        return None

    kw = keyword.strip().upper()
    cands = [x for x in data if str(x.get("AssetType", "")).lower() == "stock"]
    if not cands:
        return None

    pref_mics = set([m.strip().upper() for m in (preferred_mics or []) if m])

    best = None
    best_score = -1e18

    for x in cands:
        raw_sym = str(x.get("Symbol", "")).upper()          # e.g. "NVDA:XNAS"
        sym_root = raw_sym.split(":", 1)[0]                 # "NVDA"
        sym_mic  = raw_sym.split(":", 1)[1] if ":" in raw_sym else None  # "XNAS"
        exid = str(x.get("ExchangeId", "") or "")           # e.g. "NASDAQ"
        cur  = str(x.get("CurrencyCode", "") or "")         # e.g. "USD"
        desc = str(x.get("Description", "") or "")
        prim = bool(x.get("PrimaryListing") is True)

        score = 0.0

        # 1) Exact root symbol match is dominant
        if sym_root == kw:
            score += 120.0
        else:
            # allow some tolerance for formats / dots, etc.
            score += 10.0 * name_similarity(sym_root, kw)

        # 2) MIC preference (from your iShares exchange mapping)
        if pref_mics and sym_mic and sym_mic in pref_mics:
            score += 40.0

        # 3) Currency preference (iShares Market Currency)
        if expected_currency and cur == expected_currency:
            score += 12.0

        # 4) Primary listing
        if prim:
            score += 6.0

        # 5) Name/description similarity (iShares name vs Saxo description)
        if expected_name:
            score += 30.0 * name_similarity(expected_name, desc)

        # 6) Small boost if ExchangeId is non-empty (some results are sparse)
        if exid:
            score += 1.0

        if score > best_score:
            best_score = score
            best = x

    return best

def resolve_tickers_with_diagnostics(
    universe_batch: pd.DataFrame,   # expects at least: symbol; optional: name, saxo_market, market_currency
) -> pd.DataFrame:
    """
    Resolve tickers with Saxo and KEEP universe context for better matching + audit.

    Input columns (minimum):
      - symbol
    Optional (high value):
      - name              (iShares security name)
      - saxo_market       (MIC-like e.g. XNYS/XNAS/XETR/XPAR...)
      - market_currency   (USD/EUR/GBP...)
      - exchange_name     (raw iShares exchange label)

    Output:
      - all input columns preserved
      - plus: status, reason, symbol_exchange, uic, asset_type, exchange_id, currency, description, primary_listing
    """
    if universe_batch is None or len(universe_batch) == 0:
        return pd.DataFrame()

    if "symbol" not in universe_batch.columns:
        raise ValueError("universe_batch must contain a 'symbol' column")

    dfu = universe_batch.copy()

    # Normalize
    dfu["symbol"] = dfu["symbol"].astype(str).str.strip().str.upper()
    dfu = dfu.dropna(subset=["symbol"]).drop_duplicates(subset=["symbol"], keep="first").reset_index(drop=True)

    # Ensure optional columns exist
    for c in ["name", "saxo_market", "market_currency", "exchange_name"]:
        if c not in dfu.columns:
            dfu[c] = None

    rows = []

    for i, r in dfu.iterrows():
        t = r["symbol"]
        exp_name = r.get("name")
        exp_ccy = r.get("market_currency")
        exp_mic = r.get("saxo_market")

        print("ticker", i, ":", t, "| exp_mic=", exp_mic, "| exp_ccy=", exp_ccy)

        # Pass 1: broad search
        j = instrument_search(t, asset_types="Stock", top=25)
        data = j.get("Data", []) if isinstance(j, dict) else []

        # Choose best match using expectations
        best = pick_best_stock_match(
            keyword=t,
            data=data,
            preferred_mics=[exp_mic] if exp_mic else None,   # NOTE: if your pick uses ExchangeId, wire mic->ExchangeId later
            expected_currency=exp_ccy,
            expected_name=exp_name,
        )

        if best is None:
            rows.append({
                **r.to_dict(),
                "status": "NOT_FOUND",
                "reason": "No Stock match returned by Saxo",
                "symbol_exchange": None,
                "uic": None,
                "asset_type": None,
                "exchange_id": None,
                "currency": None,
                "description": None,
                "primary_listing": None,
            })
            continue

        exid = best.get("ExchangeId")
        rows.append({
            **r.to_dict(),
            "status": "OK",
            "reason": None,
            "symbol_exchange": f"{t}:{exid}" if exid is not None else None,
            "uic": best.get("Identifier"),
            "asset_type": best.get("AssetType"),
            "exchange_id": exid,
            "currency": best.get("CurrencyCode"),
            "description": best.get("Description"),
            "primary_listing": best.get("PrimaryListing"),
        })
    return pd.DataFrame(rows)


# ---------- NEW: update policy (add vs recreate) + per-market caches ----------
def update_market_cache(
    universe: pd.DataFrame,          # <-- DF complet (symbol + name + exchange_name + market_currency + ...)
    base_dir: str,
    market: str,
    mode: str = "recreate",          # "add" | "recreate" | "check"
    batch_size: int = 60,
) -> pd.DataFrame:
    """
    mode="add": resolve only symbols not already present in cache (any status), append, dedup
    mode="recreate": ignore existing cache and rebuild from universe
    mode="check": return the list of symbols not yet present in cache (no resolution, no save)
    Sauvegarde incrémentale par paquets (batch_size) pour éviter de tout perdre en cas d'erreur.

    universe DF must contain: 'symbol'
    Optionally used/kept: 'name', 'exchange_name', 'market_currency', etc.
    These columns are preserved and stored into the instrument cache for later matching/audit.
    """
    mode = mode.strip().lower()
    if mode not in ("add", "recreate", "check"):
        raise ValueError("mode must be 'add', 'recreate' or 'check'")

    if universe is None or len(universe) == 0:
        raise ValueError("universe is empty")

    if "symbol" not in universe.columns:
        raise ValueError("Le fichier univers doit contenir une colonne 'symbol'")

    df_uni = universe.copy()
    df_uni["symbol"] = df_uni["symbol"].astype(str).str.strip().str.upper()
    df_uni = df_uni.dropna(subset=["symbol"]).drop_duplicates(subset=["symbol"], keep="first").reset_index(drop=True)

    # (optional) normalize common fields if present
    if "market_currency" in df_uni.columns:
        df_uni["market_currency"] = df_uni["market_currency"].astype(str).str.strip().str.upper().replace({"NAN": None, "NONE": None, "": None})
    if "exchange_name" in df_uni.columns:
        df_uni["exchange_name"] = df_uni["exchange_name"].astype(str).str.strip().replace({"nan": None, "None": None, "": None})
    if "name" in df_uni.columns:
        df_uni["name"] = df_uni["name"].astype(str).str.strip().replace({"nan": None, "None": None, "": None})

    df_uni = enrich_universe_with_saxo_market(df_uni)
    symbols = df_uni["symbol"].tolist()

    df_existing = load_instrument_cache_market(base_dir, market)
    # Determine list to resolve / missing
    if mode == "recreate" or df_existing is None or len(df_existing) == 0:
        known = set()
        to_resolve = symbols
        if mode == "check":
            print(f"\u2714 [{market}] Check: cache vide/absente | Incoming={len(symbols)} | Missing={len(to_resolve)}")
            return df_uni[df_uni["symbol"].isin(to_resolve)].reset_index(drop=True)

        print(f"\u25b6 [{market}] Résolution instruments (recreate, batched)")
        # start empty but include universe columns
        df_out = pd.DataFrame(columns=list(df_uni.columns) + [
            "status","reason","uic","asset_type","exchange_id","currency","description","primary_listing","symbol_exchange"
        ])
    else:
        df_existing = df_existing.copy()
        df_existing["symbol"] = df_existing["symbol"].astype(str).str.upper()
        known = set(df_existing["symbol"].tolist())
        to_resolve = [s for s in symbols if s not in known]

        if mode == "check":
            print(f"\u2714 [{market}] Check: Known={len(known)} | Incoming={len(symbols)} | Missing={len(to_resolve)}")
            return df_uni[df_uni["symbol"].isin(to_resolve)].reset_index(drop=True)

        print(f"\u2714 [{market}] Cache chargé. Known={len(known)} | Incoming={len(symbols)} | To add={len(to_resolve)}")
        if mode == "add" and not to_resolve:
            return df_existing

        df_out = df_existing
        print(f"\u25b6 [{market}] Résolution instruments ({mode}, batched)")

    # Resolve in batches, saving after each batch
    total = len(to_resolve)
    for start in range(0, total, batch_size):
        batch_syms = to_resolve[start:start + batch_size]
        print(f"\u2026 [{market}] Batch {start//batch_size + 1} / {(total + batch_size - 1)//batch_size} ({len(batch_syms)} symbols)")

        # Universe slice for those symbols (keeps name/exchange_name/market_currency)
        df_uni_batch = df_uni[df_uni["symbol"].isin(batch_syms)].copy().reset_index(drop=True)

        try:
          # NEW: pass full universe context, not only symbols
          df_batch = resolve_tickers_with_diagnostics(df_uni_batch)
        except Exception as e: # Catch the exception explicitly
          df_out["symbol"] = df_out["symbol"].astype(str).str.upper()
          df_out = df_out.drop_duplicates(subset=["symbol"], keep="last").reset_index(drop=True)
          save_path = save_instrument_cache_market(df_out, base_dir, market)
          print("\u2716 Erreur pendant la résolution. Progress sauvegardé:", save_path)
          raise e # Re-raise the caught exception

        # Merge universe info into resolved info
        df_batch["symbol"] = df_batch["symbol"].astype(str).str.upper() # Use df_batch instead of df_res

        # Add a non-ambiguous key (symbol:exchange_id) if exchange_id present
        if "exchange_id" in df_batch.columns:
            df_batch["symbol_exchange"] = df_batch.apply(
                lambda r: f"{r['symbol']}:{r['exchange_id']}" if pd.notna(r.get("exchange_id")) else None,
                axis=1
            )
        else:
            df_batch["symbol_exchange"] = None

        df_out = pd.concat([df_out, df_batch], ignore_index=True)
        df_out["symbol"] = df_out["symbol"].astype(str).str.upper()
        df_out = df_out.drop_duplicates(subset=["symbol"], keep="last").reset_index(drop=True)

        save_path = save_instrument_cache_market(df_out, base_dir, market)
        print("\u2714 Progress sauvegardé:", save_path)

    return df_out




In [ ]:
# -------------- Clean Universe ------------------
# Uniquement si des doublons de colonnes
def clean_xy_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Nettoie un DataFrame en supprimant les colonnes *_x / *_y issues d'un merge.
    Reconstruit une colonne canonique si elle n'existe pas :
      - priorité à la colonne non suffixée si elle existe
      - sinon prend *_x, sinon *_y
    """
    df = df.copy()
    cols = list(df.columns)

    base_names = set(c[:-2] for c in cols if c.endswith("_x") or c.endswith("_y"))

    for base in base_names:
        col = base
        col_x = f"{base}_x"
        col_y = f"{base}_y"

        if col not in df.columns:
            if col_x in df.columns:
                df[col] = df[col_x]
            elif col_y in df.columns:
                df[col] = df[col_y]

        # drop suffix columns
        for c in (col_x, col_y):
            if c in df.columns:
                df.drop(columns=c, inplace=True)

    return df


def clean_instrument_cache_market(base_dir: str, market: str) -> str:
    """
    Charge le cache instruments parquet pour un marché, nettoie *_x/_y, et réécrit le parquet.
    Retourne le chemin du parquet écrit.
    """
    df = load_instrument_cache_market(base_dir, market)
    if df is None or len(df) == 0:
        print(f"✔ [{market}] cache vide, rien à nettoyer.")
        return cache_path_for_market(base_dir, market)

    before_cols = df.columns.tolist()
    df_clean = clean_xy_columns(df)

    # Normalisations minimales
    if "symbol" in df_clean.columns:
        df_clean["symbol"] = df_clean["symbol"].astype(str).str.strip().str.upper()

    # Dedup (si tu veux strict sur symbol)
    if "symbol" in df_clean.columns:
        df_clean = df_clean.drop_duplicates(subset=["symbol"], keep="last").reset_index(drop=True)

    save_path = save_instrument_cache_market(df_clean, base_dir, market)

    after_cols = df_clean.columns.tolist()
    removed = sorted(set(before_cols) - set(after_cols))
    added = sorted(set(after_cols) - set(before_cols))

    print(f"✔ [{market}] cache nettoyé: {save_path}")
    if removed:
        print("  - colonnes supprimées:", removed)
    if added:
        print("  + colonnes ajoutées:", added)

    # Guard: no more *_x/_y
    assert not any(c.endswith("_x") or c.endswith("_y") for c in df_clean.columns), "❌ Colonnes _x/_y restantes"

    return save_path

BASE_CACHE_DIR = os.environ["BASE_CACHE_DIR"]

for m in ["us", "eu"]:
    clean_instrument_cache_market(BASE_CACHE_DIR, m)


✔ [us] cache nettoyé: /content/drive/MyDrive/saxo/cache/instruments_us.parquet
  - colonnes supprimées: ['exchange_name_x', 'exchange_name_y', 'market_currency_x', 'market_currency_y', 'name_x', 'name_y', 'saxo_market_x', 'saxo_market_y']
✔ [eu] cache nettoyé: /content/drive/MyDrive/saxo/cache/instruments_eu.parquet
  - colonnes supprimées: ['exchange_name_x', 'exchange_name_y', 'market_currency_x', 'market_currency_y', 'name_x', 'name_y', 'saxo_market_x', 'saxo_market_y']


Mise à jour de la liste en cache des valeurs

Utiliser "recreate" au lieu de "add" dans la fonctions update_market_cache


In [ ]:
BASE_CACHE_DIR = os.environ["BASE_CACHE_DIR"]


# Index par marché pour compléter l'autodiscovery
seed_by_market = {c["market"].strip().lower(): c for c in markets_cfg}

# 1) Lister les univers disponibles + autodiscovery
files = list_universe_files()
print("Univers disponibles:")
for f in files:
    print(" -", os.path.basename(f))

# Convention attendue: "<universe_name>_tickers.csv"
# Ex: "us_IWB_Russell1000_tickers.csv", "eu_IWB_STOXX600_tickers.csv"
markets_cfg = []
for f in files:
    name = os.path.basename(f)
    if not name.lower().endswith("_tickers.csv"):
        continue

    base = name[:-len("_tickers.csv")]                 # ex: "eu_IWB_STOXX600"
    if "_" not in base:
        continue

    market = base.split("_", 1)[0].strip().lower()     # ex: "eu"
    if not market:
        continue

    cfg = {
        "market": market,
        "universe_name": base,                          # ex: "eu_IWB_STOXX600"
        "universe_file": name,                          # ex: "eu_IWB_STOXX600_tickers.csv"
    }

    # 2) Compléter avec les valeurs seed si elles existent (évite les conflits)
    if market in seed_by_market:
        cfg.update(seed_by_market[market])             # label, ishares_url, benchmark_symbol, update_mode, etc.

    markets_cfg.append(cfg)

# 3) Dédupe par marché (on garde une seule entrée par marché)
# Priorité: celles qui ont des champs seed (ishares_url/benchmark_symbol/label)
by_market = {}
for cfg in markets_cfg:
    m = cfg["market"]
    score = int("ishares_url" in cfg) + int("benchmark_symbol" in cfg) + int("label" in cfg)
    if m not in by_market:
        by_market[m] = (score, cfg)
    else:
        if score > by_market[m][0]:
            by_market[m] = (score, cfg)

markets_cfg = [v[1] for v in by_market.values()]
markets_cfg = sorted(markets_cfg, key=lambda x: x["market"])

print("Univers autodécouverts (complétés par seed si dispo):")
for cfg in markets_cfg:
    print(" -", cfg["market"], "|", cfg.get("universe_name"), "|", cfg.get("label"), "|", cfg.get("update_mode"))

for cfg in markets_cfg:
    print(f" - market={cfg['market']} | file={cfg['universe_file']}")

# 2) Run per market
for cfg in markets_cfg:
    market = cfg["market"]
    universe_path = os.path.join(UNIVERSE_DIR, cfg["universe_file"])
    mode = cfg.get("update_mode", "add")

    df_universe = load_universe(universe_path)
    tickers = df_universe["symbol"].tolist()
    print(f"Tickers {market} chargés:", len(tickers))

    # Build / update cache
    df_market = update_market_cache(
        df_universe,
        BASE_CACHE_DIR,
        market,
        mode   # "add" ou "recreate"
    )

    # Check unresolved (tickers absents du cache)
    df_unresolved = update_market_cache(
        df_universe,
        BASE_CACHE_DIR,
        market,
        mode="check"
    )

    if len(df_unresolved) > 0:
        df_unresolved["market"] = market
        display(df_unresolved[["market", "symbol"]])

        unresolved_path = os.path.join(UNIVERSE_DIR, f"unresolved_{market}.csv")
        df_unresolved.to_csv(unresolved_path, index=False)



Univers disponibles:
 - eu_IWB_STOXX600_tickers.csv
 - us_IWB_Russell1000_tickers.csv
Univers autodécouverts (complétés par seed si dispo):
 - eu | eu_IWB_STOXX600 | Europe | add
 - us | us_IWB_Russell1000 | United States | add
 - market=eu | file=eu_IWB_STOXX600_tickers.csv
 - market=us | file=us_IWB_Russell1000_tickers.csv
Tickers eu chargés: 551
✔ [eu] Cache chargé. Known=551 | Incoming=551 | To add=0
✔ [eu] Check: Known=551 | Incoming=551 | Missing=0
Tickers us chargés: 1008
✔ [us] Cache chargé. Known=300 | Incoming=1008 | To add=708
▶ [us] Résolution instruments (add, batched)
… [us] Batch 1 / 12 (60 symbols)
ticker 0 : SOFI | exp_mic= XNAS | exp_ccy= USD
ticker 1 : WTW | exp_mic= XNAS | exp_ccy= USD
ticker 2 : MDB | exp_mic= XNAS | exp_ccy= USD
ticker 3 : IBKR | exp_mic= XNAS | exp_ccy= USD
ticker 4 : EQT | exp_mic= XNYS | exp_ccy= USD
ticker 5 : RJF | exp_mic= XNYS | exp_ccy= USD
ticker 6 : EXR | exp_mic= XNYS | exp_ccy= USD
ticker 7 : ADM | exp_mic= XNYS | exp_ccy= USD
ticker 8

# Téléchargement et Traitement des données


## download OHLCV daily (2 ans) → Parquet, en batch (60 tickers) et sauvegarde incrémentale

In [ ]:
import pandas as pd
import numpy as np
import time

def get_account_key() -> str:
    if "SAXO_ACCOUNT_KEY" in os.environ:
        return os.environ["SAXO_ACCOUNT_KEY"]

    j = saxo_get("/port/v1/accounts/me")
    if not isinstance(j, dict) or "Data" not in j or len(j["Data"]) == 0:
        raise RuntimeError("No accounts returned by /port/v1/accounts/me")

    ak = j["Data"][0]["AccountKey"]
    os.environ["SAXO_ACCOUNT_KEY"] = ak
    return ak

def chart_get_daily(
    account_key: str,
    uic: int,
    asset_type: str,
    count: int = 300,
    time_iso: str | None = None,      # "YYYY-MM-DDTHH:MM:SSZ"
    mode: str = "UpTo",
    field_groups: str = "ChartInfo,Data",
    sample_fieldset: str = "LastTraded",    # RECOMMANDÉ (Open/High/Low/Close/Volume)
):
    """
    Wrapper Saxo OpenAPI /chart/v3/charts for DAILY bars (Horizon=1440).

    Notes:
    - Mode=UpTo REQUIRES Time
    - Horizon=1440 = daily
    - sample_fieldset MUST expose OHLC + Volume
    """

    if mode == "UpTo" and not time_iso:
        raise ValueError("Mode=UpTo requires time_iso (UTC ISO string).")

    params = {
        "AccountKey": account_key,
        "Uic": int(uic),
        "AssetType": asset_type,
        "Horizon": 1440,              # daily
        "Count": int(count),
        "Mode": mode,
        "FieldGroups": field_groups,
        "ChartSampleFieldSet": sample_fieldset,
    }
    if time_iso:
        params["Time"] = time_iso

    return saxo_get("/chart/v3/charts", params=params)


def _utc_today_floor() -> pd.Timestamp:
    return pd.Timestamp.utcnow().normalize()

def _compute_missing_bars_N(last_dt_utc: pd.Timestamp | None, safety_buffer: int = 3, max_n: int = 120) -> int:
    """
    N is computed from (today - last_date) in calendar days, with a safety buffer to cover weekends/holidays.
    If last_dt is None -> caller should treat as "new symbol" (full history).
    """
    if last_dt_utc is None or pd.isna(last_dt_utc):
        return 0
    today = _utc_today_floor()
    last_day = pd.Timestamp(last_dt_utc).tz_convert("UTC").normalize()
    delta_days = int((today - last_day).days)
    # If delta_days <= 0, still fetch a small tail for safety/dedup
    n = max(5, delta_days + safety_buffer)
    return int(min(n, max_n))
import pandas as pd

def _samples_to_df(symbol: str, uic: int, asset_type: str, resp: dict) -> pd.DataFrame:
    """
    Convert Saxo chart/v3 response into a normalized OHLCV DataFrame.

    Expected fields in resp["Data"][i]:
      - Time
      - Open, High, Low, Close
      - Volume (may be None on some instruments)

    Returns columns:
      date, symbol, uic, asset_type, open, high, low, close, volume
    """
    if not isinstance(resp, dict):
        return pd.DataFrame(columns=[
            "date","symbol","uic","asset_type","open","high","low","close","volume"
        ])

    data = resp.get("Data", [])
    if not isinstance(data, list) or len(data) == 0:
        return pd.DataFrame(columns=[
            "date","symbol","uic","asset_type","open","high","low","close","volume"
        ])

    rows = []
    for s in data:
        rows.append({
            "date": s.get("Time"),
            "symbol": symbol,
            "uic": int(uic),
            "asset_type": asset_type,
            "open": s.get("Open"),
            "high": s.get("High"),
            "low": s.get("Low"),
            "close": s.get("Close"),
            "volume": s.get("Volume"),
        })

    df = pd.DataFrame(rows)

    # Normalisation stricte
    df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
    df = df.dropna(subset=["date"])

    for c in ["open","high","low","close","volume"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = (
        df.sort_values("date")
        .drop_duplicates(subset=["date","symbol"], keep="last")
        .reset_index(drop=True)
    )

    return df

def download_daily_ohlcv_tail_for_instrument(
    account_key: str,
    symbol: str,
    uic: int,
    asset_type: str,
    n: int,
    sleep_s: float = 0.10,
) -> pd.DataFrame:
    """
    Fetch the last N daily bars (tail) using Mode=UpTo anchored to 'now'.
    Then caller filters by last_dt and appends.
    """
    if n <= 0:
        return pd.DataFrame(columns=["date","symbol","uic","asset_type","open","high","low","close","volume"])

    time_iso = pd.Timestamp.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    resp = chart_get_daily(
        account_key=account_key,
        uic=int(uic),
        asset_type=str(asset_type),
        count=int(n),
        time_iso=time_iso,
        mode="UpTo",
        # sample_fieldset should be "OHLC" (recommended). If you keep LastTraded, ensure Open/High/Low/Close/Volume exist.
        # sample_fieldset="OHLC",
    )
    time.sleep(sleep_s)
    return _samples_to_df(symbol, uic, asset_type, resp)


def ohlcv_cache_path_for_market(base_dir: str, market: str) -> str:
    """
    Canonical path for OHLCV daily cache per market.
    """
    market = market.strip().lower()
    os.makedirs(base_dir, exist_ok=True)
    return os.path.join(base_dir, f"ohlcv_daily_{market}.parquet")


def load_ohlcv_cache_market(base_dir: str, market: str) -> pd.DataFrame | None:
    """
    Load OHLCV daily cache for a given market.
    Returns None if file does not exist.
    """
    path = ohlcv_cache_path_for_market(base_dir, market)
    if not os.path.exists(path):
        return None

    df = pd.read_parquet(path)

    # Minimal normalization (safety)
    if "symbol" in df.columns:
        df["symbol"] = df["symbol"].astype(str).str.upper()
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")

    return df


def save_ohlcv_cache_market(df: pd.DataFrame, base_dir: str, market: str) -> str:
    """
    Save OHLCV daily cache for a given market.
    """
    path = ohlcv_cache_path_for_market(base_dir, market)
    df.to_parquet(path, index=False)
    return path

def update_ohlcv_market_cache_batched(
    df_instruments_ok: pd.DataFrame,
    base_dir: str,
    market: str,
    mode: str = "add",         # "add" or "recreate"
    batch_size: int = 60,
    years: int = 2,
    safety_buffer: int = 3,    # extra bars to cover non-trading days
    max_tail_n: int = 120,     # cap on tail fetch size
):
    """
    NEW BEHAVIOR (mode="add"):
      - For symbols already in the OHLCV parquet: compute last downloaded date and fetch only missing tail (N bars).
      - For new symbols: fetch ~years of history (existing behavior).

    Saves progress after each batch.
    """
    mode = mode.strip().lower()
    if mode not in ("add", "recreate"):
        raise ValueError("mode must be 'add' or 'recreate'")

    needed = df_instruments_ok.copy()
    needed["symbol"] = needed["symbol"].astype(str).str.upper()
    needed = needed.dropna(subset=["symbol", "uic", "asset_type"]).drop_duplicates(subset=["symbol"]).reset_index(drop=True)

    df_existing = load_ohlcv_cache_market(base_dir, market)

    if mode == "recreate" or df_existing is None or len(df_existing) == 0:
        df_out = pd.DataFrame(columns=["date","symbol","uic","asset_type","open","high","low","close","volume"])
        print(f"▶ [{market}] OHLCV download (recreate, batched)")
        # recreate = treat everyone as new
        last_date_by_symbol = {}
    else:
        df_out = df_existing.copy()
        df_out["symbol"] = df_out["symbol"].astype(str).str.upper()
        df_out["date"] = pd.to_datetime(df_out["date"], utc=True, errors="coerce")
        last_date_by_symbol = df_out.groupby("symbol")["date"].max().to_dict()
        print(f"✔ [{market}] OHLCV cache chargé. Symbols={len(last_date_by_symbol)}")

    account_key = get_account_key()

    total = len(needed)
    for start in range(0, total, batch_size):
        batch = needed.iloc[start:start + batch_size]
        print(f"… [{market}] Batch {start//batch_size + 1} / {(total + batch_size - 1)//batch_size} ({len(batch)} symbols)")

        parts = []
        try:
            for _, row in batch.iterrows():
                sym = row["symbol"]
                uic = int(row["uic"])
                at  = str(row["asset_type"])

                last_dt = last_date_by_symbol.get(sym)
                if mode == "add" and last_dt is not None and not pd.isna(last_dt):
                    # Incremental tail fetch
                    n = _compute_missing_bars_N(last_dt, safety_buffer=safety_buffer, max_n=max_tail_n)
                    df_sym = download_daily_ohlcv_tail_for_instrument(
                        account_key=account_key,
                        symbol=sym,
                        uic=uic,
                        asset_type=at,
                        n=n,
                    )
                    if not df_sym.empty:
                        df_sym = df_sym[df_sym["date"] > pd.Timestamp(last_dt).tz_convert("UTC")].copy()
                    print(f"Incremental downloaded: {sym} N={n} new_rows={len(df_sym)} last_dt={last_dt}")
                else:
                    # New symbol (or recreate): full history window
                    df_sym = download_daily_ohlcv_for_instrument(
                        account_key=account_key,
                        symbol=sym,
                        uic=uic,
                        asset_type=at,
                        years=years,
                    )
                    print(f"Full history downloaded: {sym} rows={len(df_sym)}")

                if df_sym is not None and len(df_sym) > 0:
                    parts.append(df_sym)

        except Exception:
            df_out = df_out.drop_duplicates(subset=["date","symbol"], keep="last").reset_index(drop=True)
            save_path = save_ohlcv_cache_market(df_out, base_dir, market)
            print("✖ Erreur pendant OHLCV. Progress sauvegardé:", save_path)
            raise

        if parts:
            df_batch = pd.concat(parts, ignore_index=True)
            df_out = pd.concat([df_out, df_batch], ignore_index=True)
            df_out["symbol"] = df_out["symbol"].astype(str).str.upper()
            df_out["date"] = pd.to_datetime(df_out["date"], utc=True, errors="coerce")
            df_out = df_out.drop_duplicates(subset=["date","symbol"], keep="last").reset_index(drop=True)

            # update last_date_by_symbol incrementally
            upd = df_batch.copy()
            upd["symbol"] = upd["symbol"].astype(str).str.upper()
            upd["date"] = pd.to_datetime(upd["date"], utc=True, errors="coerce")
            upd_last = upd.groupby("symbol")["date"].max().to_dict()
            last_date_by_symbol.update(upd_last)

        save_path = save_ohlcv_cache_market(df_out, base_dir, market)
        print("✔ Progress sauvegardé:", save_path)

    return df_out

In [ ]:
BASE_CACHE_DIR = os.environ["BASE_CACHE_DIR"]

# 1) Combine all instrument caches (per market) into one df
markets = sorted({cfg["market"] for cfg in markets_cfg})
df_cache_all = combine_market_caches(BASE_CACHE_DIR, markets=markets)

# 2) Keep only OK instruments + normalize
df_ok_all = df_cache_all[df_cache_all["status"] == "OK"][["market", "symbol", "uic", "asset_type"]].copy()
df_ok_all["market"] = df_ok_all["market"].astype(str).str.lower()
df_ok_all["symbol"] = df_ok_all["symbol"].astype(str).str.upper()

# 3) Download OHLCV per market in a loop
ohlcv_by_market = {}

for m in markets:
    df_ok_m = (
        df_ok_all[df_ok_all["market"] == m][["symbol", "uic", "asset_type"]]
        .drop_duplicates(subset=["symbol"])
        .reset_index(drop=True)
    )

    if len(df_ok_m) == 0:
        print(f"⚠️ [{m}] Aucun instrument OK, skip.")
        continue

    ohlcv_by_market[m] = update_ohlcv_market_cache_batched(
        df_ok_m,
        BASE_CACHE_DIR,
        market=m,
        mode="add",
        batch_size=60,
        years=2
    )


# Access results e.g.:
# df_ohlcv_us = ohlcv_by_market["us"]
# df_ohlcv_eu = ohlcv_by_market["eu"]

✔ [eu] OHLCV cache chargé. Symbols=522
… [eu] Batch 1 / 9 (60 symbols)
Incremental downloaded: ASML N=5 new_rows=0 last_dt=2026-01-21 00:00:00+00:00
Incremental downloaded: ROG N=5 new_rows=0 last_dt=2026-01-21 00:00:00+00:00
Incremental downloaded: AZN N=5 new_rows=0 last_dt=2026-01-21 00:00:00+00:00
Incremental downloaded: HSBA N=5 new_rows=0 last_dt=2026-01-21 00:00:00+00:00
Incremental downloaded: NOVN N=5 new_rows=0 last_dt=2026-01-21 00:00:00+00:00
Incremental downloaded: SAP N=5 new_rows=0 last_dt=2026-01-20 00:00:00+00:00
Incremental downloaded: NESN N=5 new_rows=0 last_dt=2026-01-21 00:00:00+00:00
Incremental downloaded: SIE N=5 new_rows=0 last_dt=2026-01-21 00:00:00+00:00
Incremental downloaded: SHELL N=5 new_rows=0 last_dt=2026-01-21 00:00:00+00:00
Incremental downloaded: MC N=5 new_rows=0 last_dt=2026-01-20 00:00:00+00:00
Incremental downloaded: SAN N=5 new_rows=0 last_dt=2026-01-21 00:00:00+00:00
Incremental downloaded: ALV N=5 new_rows=0 last_dt=2026-01-21 00:00:00+00:00


## calcul des indicateurs

**Feature Store**
Objectif : calculer une fois les indicateurs nécessaires, les stocker, et ne plus recalculer à la volée.

À calculer (par symbole) :
* Momentum 12–1 (proxy daily)
* Proximité du plus haut 52 semaines
* Tendance
	* MM50, MM150, MM200
  * Flags : close>MM150, MM50>MM150, MM150>MM200
* Breakout
	* close > max(high, 20j) et/ou 50j
* Volume relatif (RVOL)
	*	volume / mean(volume, 20j)
*	ATR(14) (pour le stop et le sizing)

Livrable : un Parquet features_daily_{market}.parquet.

In [ ]:
import os
import numpy as np
import pandas as pd

# --------- Feature store paths (per market) ---------

def features_cache_path_for_market(base_dir: str, market: str) -> str:
    market = market.strip().lower()
    os.makedirs(base_dir, exist_ok=True)
    return os.path.join(base_dir, f"features_daily_{market}.parquet")

def load_features_cache_market(base_dir: str, market: str) -> pd.DataFrame | None:
    path = features_cache_path_for_market(base_dir, market)
    if os.path.exists(path):
        return pd.read_parquet(path)
    return None

def save_features_cache_market(df: pd.DataFrame, base_dir: str, market: str) -> str:
    path = features_cache_path_for_market(base_dir, market)
    df.to_parquet(path, index=False)
    return path

# --------- Core feature computation ---------

def _atr14(df: pd.DataFrame) -> pd.Series:
    # True Range
    prev_close = df["close"].shift(1)
    tr = pd.concat([
        (df["high"] - df["low"]).abs(),
        (df["high"] - prev_close).abs(),
        (df["low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.rolling(14, min_periods=14).mean()

def compute_features_daily(df_ohlcv: pd.DataFrame) -> pd.DataFrame:
    """
    Input columns required:
      date, symbol, open, high, low, close, volume
    Output: daily feature rows per symbol/date (only where enough history exists)
    """
    df = df_ohlcv.copy()

    # Basic hygiene
    df["symbol"] = df["symbol"].astype(str).str.upper()
    df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
    df = df.dropna(subset=["date", "symbol", "close"]).sort_values(["symbol", "date"]).reset_index(drop=True)

    # Ensure numeric
    for c in ["open", "high", "low", "close", "volume"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    feats = []
    for sym, g in df.groupby("symbol", sort=False):
        g = g.sort_values("date").reset_index(drop=True)

        # Moving averages
        g["sma50"]  = g["close"].rolling(50,  min_periods=50).mean()
        g["sma150"] = g["close"].rolling(150, min_periods=150).mean()
        g["sma200"] = g["close"].rolling(200, min_periods=200).mean()

        # Momentum proxy (12-1): (252d return) - (21d return)
        g["ret_252"] = g["close"] / g["close"].shift(252) - 1.0
        g["ret_21"]  = g["close"] / g["close"].shift(21)  - 1.0
        g["mom_12_1"] = g["ret_252"] - g["ret_21"]

        # 52-week high proximity
        g["high_52w"] = g["close"].rolling(252, min_periods=252).max()
        g["near_52w"] = g["close"] / g["high_52w"]

        # Breakouts
        g["hh20"] = g["high"].rolling(20, min_periods=20).max()
        g["hh50"] = g["high"].rolling(50, min_periods=50).max()
        g["breakout_20"] = g["close"] > g["hh20"].shift(1)
        g["breakout_50"] = g["close"] > g["hh50"].shift(1)

        # RVOL
        g["vol_ma20"] = g["volume"].rolling(20, min_periods=20).mean()
        g["rvol20"] = g["volume"] / g["vol_ma20"]

        # ATR(14)
        g["atr14"] = _atr14(g)

        # Trend flags
        g["trend_close_gt_sma150"] = g["close"] > g["sma150"]
        g["trend_sma50_gt_sma150"] = g["sma50"] > g["sma150"]
        g["trend_sma150_gt_sma200"] = g["sma150"] > g["sma200"]

        # Keep only useful columns
        out = g[[
            "date","symbol","open","high","low","close","volume",
            "sma50","sma150","sma200",
            "mom_12_1","ret_252","ret_21",
            "high_52w","near_52w",
            "hh20","hh50","breakout_20","breakout_50",
            "rvol20","atr14",
            "trend_close_gt_sma150","trend_sma50_gt_sma150","trend_sma150_gt_sma200"
        ]].copy()

        feats.append(out)

    df_feat = pd.concat(feats, ignore_index=True) if feats else pd.DataFrame()

    # Drop rows without the minimum history (choose strict: need 252d close + 200d SMA)
    min_ok = (
        df_feat["mom_12_1"].notna()
        & df_feat["near_52w"].notna()
        & df_feat["sma200"].notna()
    )
    df_feat = df_feat[min_ok].sort_values(["symbol","date"]).reset_index(drop=True)
    return df_feat

# --------- Incremental builder (append only new dates) ---------

def update_features_market_cache(
    df_ohlcv_market: pd.DataFrame,
    base_dir: str,
    market: str,
    mode: str = "add",          # "add" or "recreate"
) -> pd.DataFrame:
    """
    mode="add": compute features for new dates only (per symbol), append & dedup
    mode="recreate": compute features for all OHLCV rows
    """
    mode = mode.strip().lower()
    if mode not in ("add","recreate"):
        raise ValueError("mode must be 'add' or 'recreate'")

    df_ohlcv_market = df_ohlcv_market.copy()
    df_ohlcv_market["symbol"] = df_ohlcv_market["symbol"].astype(str).str.upper()
    df_ohlcv_market["date"] = pd.to_datetime(df_ohlcv_market["date"], utc=True, errors="coerce")
    df_ohlcv_market = df_ohlcv_market.dropna(subset=["date","symbol"]).sort_values(["symbol","date"])

    df_existing = load_features_cache_market(base_dir, market)

    if mode == "recreate" or df_existing is None or len(df_existing) == 0:
        print(f"▶ [{market}] Features (recreate)")
        df_feat = compute_features_daily(df_ohlcv_market)
        save_path = save_features_cache_market(df_feat, base_dir, market)
        print("✔ Features écrites:", save_path)
        return df_feat

    # add-mode: compute only dates after last feature date per symbol,
    # but include a lookback window (max 260 rows) to make rolling indicators correct.
    df_existing["symbol"] = df_existing["symbol"].astype(str).str.upper()
    df_existing["date"] = pd.to_datetime(df_existing["date"], utc=True, errors="coerce")
    last_by_sym = df_existing.groupby("symbol")["date"].max().to_dict()

    parts = []
    for sym, g in df_ohlcv_market.groupby("symbol", sort=False):
        g = g.sort_values("date").reset_index(drop=True)
        last_dt = last_by_sym.get(sym)

        if last_dt is None:
            # New symbol: compute all (but only last ~2y is already in OHLCV)
            g_in = g
        else:
            # Need enough history for rolling windows; keep tail up to 260 rows before last_dt
            # and all rows after last_dt
            idx_last = g.index[g["date"] <= last_dt]
            if len(idx_last) == 0:
                g_in = g
            else:
                cut = int(idx_last.max())
                start = max(0, cut - 260)  # ensures 252/200 windows are correct
                g_in = g.iloc[start:].copy()

        parts.append(g_in)

    df_in = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    print(f"▶ [{market}] Features (add) on {df_in['symbol'].nunique()} symbols")

    df_new = compute_features_daily(df_in)

    # Keep only truly new dates vs existing (per symbol)
    # The following line was incorrect and caused the TypeError:
    # df_new = df_new[~list(zip(df_new["symbol"], df_new["date"])).__iter__()]  # placeholder to keep structure
    # Efficient filter:
    df_new = df_new[~pd.MultiIndex.from_frame(df_new[["symbol","date"]]).isin(
        pd.MultiIndex.from_frame(df_existing[["symbol","date"]])
    )]

    df_out = pd.concat([df_existing, df_new], ignore_index=True)
    df_out = df_out.drop_duplicates(subset=["symbol","date"], keep="last").sort_values(["symbol","date"]).reset_index(drop=True)

    save_path = save_features_cache_market(df_out, base_dir, market)
    print("✔ Features mises à jour:", save_path)
    return df_out

def compute_market_regime_from_ohlcv(df_ohlcv: pd.DataFrame) -> dict:
    d = df_ohlcv.copy()
    d["date"] = pd.to_datetime(d["date"], utc=True)
    d = d.sort_values("date")

    d["sma200"] = d["close"].rolling(200).mean()
    last = d.iloc[-1]

    return {
        "date": last["date"],
        "close": float(last["close"]),
        "sma200": float(last["sma200"]),
        "risk_on": bool(last["close"] > last["sma200"]),
    }

In [ ]:
# === EXECUTION: Feature store (per market) using results from previous OHLCV loop ===
# Preconditions:
#  - ohlcv_by_market dict exists from the OHLCV loop: {market: df_ohlcv_market}
#  - update_features_market_cache(...) is defined

BASE_CACHE_DIR = os.environ["BASE_CACHE_DIR"]

feat_by_market = {}

for m, df_ohlcv_m in ohlcv_by_market.items():
    if df_ohlcv_m is None or len(df_ohlcv_m) == 0:
        print(f"⚠️ [{m}] OHLCV vide, skip features.")
        continue

    feat_by_market[m] = update_features_market_cache(
        df_ohlcv_market=df_ohlcv_m,
        base_dir=BASE_CACHE_DIR,
        market=m,
        mode="add"   # "recreate" to rebuild everything
    )

    df_feat_m = feat_by_market[m]
    last_dt = df_feat_m["date"].max() if len(df_feat_m) else None
    print(f"✔ [{m}] features: {len(df_feat_m)} rows | last date: {last_dt}")

# Access results e.g.:
# df_feat_us = feat_by_market["us"]
# df_feat_eu = feat_by_market["eu"]

# Optional: load later without recompute
# df_feat_us = load_features_cache_market(BASE_CACHE_DIR, "us")
# df_feat_eu = load_features_cache_market(BASE_CACHE_DIR, "eu")

▶ [eu] Features (add) on 522 symbols
✔ Features mises à jour: /content/drive/MyDrive/saxo/cache/features_daily_eu.parquet
✔ [eu] features: 131484 rows | last date: 2026-01-21 00:00:00+00:00
▶ [us] Features (add) on 981 symbols
✔ Features mises à jour: /content/drive/MyDrive/saxo/cache/features_daily_us.parquet
✔ [us] features: 366291 rows | last date: 2026-01-21 00:00:00+00:00


## Scoring


**Étape suivante : scoring + shortlist (signaux d’achat exploitables)**

Tu as maintenant features_daily_{market}.parquet. La prochaine brique consiste à :

	1.	calculer des rangs/percentiles (momentum, near 52W high)
	2.	appliquer les filtres trend + breakout + volume
	3.	produire un score /100 et une shortlist (top N) exportable

Sorties attendues

* signals_daily_{market}.parquet (toutes les lignes avec score)
* shortlist_{market}.csv (candidats du jour)




In [ ]:
import os
import numpy as np
import pandas as pd

# ---------- Paths ----------

def signals_cache_path_for_market(base_dir: str, market: str) -> str:
    market = market.strip().lower()
    os.makedirs(base_dir, exist_ok=True)
    return os.path.join(base_dir, f"signals_daily_{market}.parquet")

def shortlist_path_for_market(base_dir: str, market: str) -> str:
    market = market.strip().lower()
    os.makedirs(base_dir, exist_ok=True)
    return os.path.join(base_dir, f"shortlist_{market}.csv")

def save_signals_market(df: pd.DataFrame, base_dir: str, market: str) -> str:
    path = signals_cache_path_for_market(base_dir, market)
    df.to_parquet(path, index=False)
    return path

def load_signals_market(base_dir: str, market: str) -> pd.DataFrame | None:
    path = signals_cache_path_for_market(base_dir, market)
    if not os.path.exists(path):
        return None
    return pd.read_parquet(path)

# ---------- Utilities ----------

def _percentile_rank(s: pd.Series) -> pd.Series:
    # robust percentile: 0..1; handles ties
    return s.rank(pct=True, method="average")

def _clamp01(x: pd.Series) -> pd.Series:
    return x.clip(lower=0.0, upper=1.0)

def _bool_to_int(x: pd.Series) -> pd.Series:
    return x.astype(bool).astype(int)


# ---------- Scoring core ----------

def score_features_for_all_dates(
    df_feat: pd.DataFrame,
    top_n: int = 50,
    score_threshold: int = 70,
    rvol_threshold: float = 1.2,
    use_breakout: str = "50",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Calcule les signaux pour TOUTES les dates disponibles dans df_feat.
    Retourne (df_signals_all, df_shortlist_all) concaténés sur toutes les dates.
    """
    df = df_feat.copy()
    df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
    df = df.dropna(subset=["date", "symbol"])

    all_dates = sorted(df["date"].dropna().unique())
    if len(all_dates) == 0:
        raise ValueError("No dates in df_feat")

    signals_list = []
    shortlist_list = []

    for d in all_dates:
        sig_d, sh_d = score_features_for_date(
            df_feat=df,
            asof_date=d,
            top_n=top_n,
            score_threshold=score_threshold,
            rvol_threshold=rvol_threshold,
            use_breakout=use_breakout,
        )
        signals_list.append(sig_d)
        shortlist_list.append(sh_d)

    df_signals_all = pd.concat(signals_list, ignore_index=True) if signals_list else pd.DataFrame()
    df_shortlist_all = pd.concat(shortlist_list, ignore_index=True) if shortlist_list else pd.DataFrame()
    return df_signals_all, df_shortlist_all

def score_features_for_date(
    df_feat: pd.DataFrame,
    asof_date: pd.Timestamp | str | None = None,
    top_n: int = 50,
    score_threshold: int = 70,
    rvol_threshold: float = 1.2,
    use_breakout: str = "50",  # "20" or "50"
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns (df_signals_asof, df_shortlist_asof)

    Required columns in df_feat:
      date,symbol,close,volume,
      mom_12_1, near_52w,
      breakout_20, breakout_50,
      rvol20,
      trend_close_gt_sma150, trend_sma50_gt_sma150, trend_sma150_gt_sma200
    """
    df = df_feat.copy()
    df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
    df["symbol"] = df["symbol"].astype(str).str.upper()
    df = df.dropna(subset=["date","symbol"])

    if asof_date is None:
        asof = df["date"].max()
    else:
        asof = pd.to_datetime(asof_date, utc=True)
        # pick latest available <= asof
        asof = df.loc[df["date"] <= asof, "date"].max()

    if pd.isna(asof):
        raise ValueError("No data available for the requested asof_date")

    dfd = df[df["date"] == asof].copy()
    dfd = dfd.dropna(subset=["mom_12_1","near_52w"])

    # Percentiles across universe that day
    dfd["mom_pct"] = _percentile_rank(dfd["mom_12_1"])
    dfd["near52_pct"] = _percentile_rank(dfd["near_52w"])

    # Binary filters
    if use_breakout == "20":
        dfd["breakout_ok"] = _bool_to_int(dfd["breakout_20"])
    else:
        dfd["breakout_ok"] = _bool_to_int(dfd["breakout_50"])

    dfd["rvol_ok"] = (pd.to_numeric(dfd["rvol20"], errors="coerce") >= float(rvol_threshold)).astype(int)

    dfd["trend_ok"] = (
        _bool_to_int(dfd["trend_close_gt_sma150"])
        & _bool_to_int(dfd["trend_sma50_gt_sma150"])
        & _bool_to_int(dfd["trend_sma150_gt_sma200"])
    ).astype(int)

    # Continuous components (0..1)
    # near_52w typically ~0.7..1.0 ; we cap to [0.75..1.0] mapped to [0..1] to avoid over-rewarding weak proximity.
    near_raw = pd.to_numeric(dfd["near_52w"], errors="coerce")
    dfd["near52_scaled"] = _clamp01((near_raw - 0.75) / (1.00 - 0.75))

    # Score weights (sum 100)
    #  - Momentum percentile: 35
    #  - Near 52W high (scaled): 20
    #  - Trend structure: 20
    #  - Breakout: 15
    #  - RVOL confirmation: 10
    dfd["score"] = (
        35.0 * dfd["mom_pct"].fillna(0.0)
        + 20.0 * dfd["near52_scaled"].fillna(0.0)
        + 20.0 * dfd["trend_ok"].fillna(0.0)
        + 15.0 * dfd["breakout_ok"].fillna(0.0)
        + 10.0 * dfd["rvol_ok"].fillna(0.0)
    ).round(0).astype(int)

    # Human-readable reasons (compact)
    def _reasons(row) -> str:
        r = []
        if row["trend_ok"] == 0: r.append("trend")
        if row["breakout_ok"] == 0: r.append(f"bo{use_breakout}")
        if row["rvol_ok"] == 0: r.append("rvol")
        if row["near52_scaled"] < 0.25: r.append("52w")
        if row["mom_pct"] < 0.70: r.append("mom")
        return ",".join(r) if r else "OK"

    dfd["reasons"] = dfd.apply(_reasons, axis=1)

    # Sort / shortlist
    dfd = dfd.sort_values(["score","mom_pct","near_52w"], ascending=[False,False,False]).reset_index(drop=True)

    shortlist = dfd[dfd["score"] >= int(score_threshold)].head(int(top_n)).copy()

    # Keep useful columns for downstream
    keep_cols = [
        "date","symbol","score","reasons",
        "close","volume",
        "mom_12_1","mom_pct",
        "near_52w","near52_scaled",
        "trend_ok","breakout_ok","rvol_ok",
        "rvol20",
        "sma50","sma150","sma200",
        "atr14",
    ]
    keep_cols = [c for c in keep_cols if c in dfd.columns]
    dfd_out = dfd[keep_cols].copy()
    shortlist_out = shortlist[keep_cols].copy()

    return dfd_out, shortlist_out


def save_shortlist_market_append(df_new: pd.DataFrame, base_dir: str, market: str) -> str:
    """
    Append the new shortlist at the TOP of the existing shortlist file (history kept).
    Do nothing if this day's shortlist has already been appended (prevents duplicates).

    Dedup key: (date, symbol). This guarantees idempotency even if you rerun the notebook.
    """
    path = shortlist_path_for_market(base_dir, market)
    os.makedirs(os.path.dirname(path), exist_ok=True)

    if df_new is None or len(df_new) == 0:
        # Still ensure file exists? We'll just no-op.
        return path

    df_new = df_new.copy()
    df_new["date"] = pd.to_datetime(df_new["date"], utc=True, errors="coerce")
    df_new["symbol"] = df_new["symbol"].astype(str).str.upper()
    df_new = df_new.dropna(subset=["date","symbol"])

    latest_date = df_new["date"].max()
    if pd.isna(latest_date):
        return path

    # Load existing history if any
    if os.path.exists(path):
        df_old = pd.read_csv(path)
        if "date" in df_old.columns:
            df_old["date"] = pd.to_datetime(df_old["date"], utc=True, errors="coerce")
        if "symbol" in df_old.columns:
            df_old["symbol"] = df_old["symbol"].astype(str).str.upper()
    else:
        df_old = pd.DataFrame(columns=df_new.columns)

    # If this date is already present, do nothing (idempotent run)
    if len(df_old) > 0 and "date" in df_old.columns and (df_old["date"].max() == latest_date):
        # Stronger check: ensure that at least one key from df_new already exists
        old_keys = set(zip(df_old["date"], df_old["symbol"])) if {"date","symbol"}.issubset(df_old.columns) else set()
        new_keys = set(zip(df_new["date"], df_new["symbol"]))
        if len(old_keys.intersection(new_keys)) > 0:
            print(f"✔ shortlist [{market}] déjà ajoutée pour {latest_date.date()} -> no-op")
            return path

    # Prepend + dedup on (date,symbol)
    df_combined = pd.concat([df_new, df_old], ignore_index=True)

    if {"date","symbol"}.issubset(df_combined.columns):
        df_combined = df_combined.drop_duplicates(subset=["date","symbol"], keep="first")

    # Sort newest first (date desc, score desc if present)
    sort_cols = ["date"]
    ascending = [False]
    if "score" in df_combined.columns:
        sort_cols.append("score")
        ascending.append(False)

    df_combined = df_combined.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)
    df_combined.to_csv(path, index=False)

    print(f"✔ shortlist [{market}] append OK: {path} (added date {latest_date.date()})")
    return path

def _enrich_with_instrument_meta(
    df: pd.DataFrame,
    base_dir: str,
    market: str,
    market_code_col: str = "exchange_id",   # expected in instrument cache (from tickers exchange mapping)
    name_col: str = "description",                 # expected in instrument cache (from tickers parquet/csv)
    symbol_exchange_col: str = "symbol_exchange",                 # expected in instrument cache (from tickers parquet/csv)
) -> pd.DataFrame:
    """
    Adds 2 columns to df (if available in instrument cache):
      - market_code (ex: XNAS, XNYS, XETR, ...)
      - name        (security name from your universe/tickers parquet/csv)

    Source: instrument cache parquet for this market.
    """
    out = df.copy()
    if out is None or len(out) == 0 or "symbol" not in out.columns:
        return out

    out["symbol"] = out["symbol"].astype(str).str.upper()

    df_ic = load_instrument_cache_market(base_dir, market)
    if df_ic is None or len(df_ic) == 0 or "symbol" not in df_ic.columns:
        out["market_code"] = None
        out["name"] = None
        out["symbol_exchange"] = None
        return out

    df_ic = df_ic.copy()
    df_ic["symbol"] = df_ic["symbol"].astype(str).str.upper()

    # Prefer explicit MIC-like code if present; fallback: parse MIC from Saxo "Symbol" if present
    mic = None
    if market_code_col in df_ic.columns:
        mic = df_ic[market_code_col]
    elif "Symbol" in df_ic.columns:  # in case you stored raw Saxo Symbol later
        mic = df_ic["Symbol"].astype(str).str.upper().str.split(":", n=1).str[1]
    elif "symbol_saxo" in df_ic.columns:
        mic = df_ic["symbol_saxo"].astype(str).str.upper().str.split(":", n=1).str[1]

    df_meta = pd.DataFrame({"symbol": df_ic["symbol"]})
    df_meta["market_code"] = mic if mic is not None else None
    df_meta["name"] = df_ic[name_col].astype(str) if name_col in df_ic.columns else None
    df_meta["symbol_exchange"] = df_ic[symbol_exchange_col].astype(str) if symbol_exchange_col in df_ic.columns else None

    # Clean placeholders
    for c in ["market_code", "name"]:
        if c in df_meta.columns:
            df_meta[c] = df_meta[c].replace({"nan": None, "None": None, "": None})

    df_meta = df_meta.drop_duplicates(subset=["symbol"], keep="last")

    out = out.merge(df_meta[["symbol", "market_code", "name"]], on="symbol", how="left")
    return out

def run_scoring_and_export(
    df_feat_market: pd.DataFrame,
    base_dir: str,
    market: str,
    asof_date: pd.Timestamp | str | None = None,
    top_n: int = 50,
    score_threshold: int = 70,
    rvol_threshold: float = 1.2,
    use_breakout: str = "50",
    reprocess_mode: str = "incremental",   # NEW: "incremental" | "full" | "full_append"
):
    reprocess_mode = (reprocess_mode or "incremental").strip().lower()
    if reprocess_mode not in {"incremental", "full", "full_append"}:
        raise ValueError("reprocess_mode must be: incremental | full | full_append")

    # --------- FULL recompute ---------
    if reprocess_mode in {"full", "full_append"}:
        df_signals, df_shortlist = score_features_for_all_dates(
            df_feat=df_feat_market,
            top_n=top_n,
            score_threshold=score_threshold,
            rvol_threshold=rvol_threshold,
            use_breakout=use_breakout,
        )

        df_signals = _enrich_with_instrument_meta(df_signals, base_dir, market)
        df_shortlist = _enrich_with_instrument_meta(df_shortlist, base_dir, market)

        # reorder (as before)
        def _reorder(df: pd.DataFrame) -> pd.DataFrame:
            if df is None or len(df) == 0:
                return df
            front = [c for c in ["date", "symbol", "market_code", "name", "score", "reasons"] if c in df.columns]
            rest = [c for c in df.columns if c not in front]
            return df[front + rest]

        df_signals = _reorder(df_signals)
        df_shortlist = _reorder(df_shortlist)

        # write parquet according to mode
        if reprocess_mode == "full":
            sig_path = save_signals_market(df_signals, base_dir, market)
        else:
            # merge with existing parquet, keep newest calculation on (date,symbol)
            df_old = load_signals_market(base_dir, market)
            if df_old is None or len(df_old) == 0:
                sig_path = save_signals_market(df_signals, base_dir, market)
            else:
                df_old = df_old.copy()
                if "date" in df_old.columns:
                    df_old["date"] = pd.to_datetime(df_old["date"], utc=True, errors="coerce")
                if "symbol" in df_old.columns:
                    df_old["symbol"] = df_old["symbol"].astype(str).str.upper()

                df_combined = pd.concat([df_signals, df_old], ignore_index=True)
                if {"date", "symbol"}.issubset(df_combined.columns):
                    df_combined = df_combined.drop_duplicates(subset=["date", "symbol"], keep="first")
                df_combined = df_combined.sort_values(["date", "score"], ascending=[False, False]) \
                                         if {"date","score"}.issubset(df_combined.columns) \
                                         else df_combined.sort_values(["date"], ascending=[False])
                sig_path = save_signals_market(df_combined.reset_index(drop=True), base_dir, market)
                df_signals = df_combined.reset_index(drop=True)

        # shortlist: tu gardes déjà un append historique — ok
        sh_path = save_shortlist_market_append(df_shortlist, base_dir, market)

        print("✔ signals:", sig_path, f"({len(df_signals)} lignes, dates={df_signals['date'].nunique()})")
        print("✔ shortlist (history):", sh_path, f"(new rows={len(df_shortlist)})")
        return df_signals, df_shortlist

    # --------- INCREMENTAL (existing behavior) ---------
    df_signals, df_shortlist = score_features_for_date(
        df_feat=df_feat_market,
        asof_date=asof_date,
        top_n=top_n,
        score_threshold=score_threshold,
        rvol_threshold=rvol_threshold,
        use_breakout=use_breakout,
    )

    df_signals = _enrich_with_instrument_meta(df_signals, base_dir, market)
    df_shortlist = _enrich_with_instrument_meta(df_shortlist, base_dir, market)

    def _reorder(df: pd.DataFrame) -> pd.DataFrame:
        if df is None or len(df) == 0:
            return df
        front = [c for c in ["date", "symbol", "market_code", "name", "score", "reasons"] if c in df.columns]
        rest = [c for c in df.columns if c not in front]
        return df[front + rest]

    df_signals = _reorder(df_signals)
    df_shortlist = _reorder(df_shortlist)

    sig_path = save_signals_market(df_signals, base_dir, market)
    sh_path = save_shortlist_market_append(df_shortlist, base_dir, market)

    print("✔ signals:", sig_path, f"({len(df_signals)} lignes, date={df_signals['date'].max()})")
    print("✔ shortlist (history):", sh_path, f"(new rows={len(df_shortlist)}, seuil={score_threshold})")
    return df_signals, df_shortlist


def run_scoring_and_export_old(
    df_feat_market: pd.DataFrame,
    base_dir: str,
    market: str,
    asof_date: pd.Timestamp | str | None = None,
    top_n: int = 50,
    score_threshold: int = 70,
    rvol_threshold: float = 1.2,
    use_breakout: str = "50",
):
    df_signals, df_shortlist = score_features_for_date(
        df_feat=df_feat_market,
        asof_date=asof_date,
        top_n=top_n,
        score_threshold=score_threshold,
        rvol_threshold=rvol_threshold,
        use_breakout=use_breakout,
    )

    # NEW: enrich outputs with market_code + name from instrument cache parquet
    df_signals = _enrich_with_instrument_meta(df_signals, base_dir, market)
    df_shortlist = _enrich_with_instrument_meta(df_shortlist, base_dir, market)

    # (optional) reorder: put market_code + name near symbol
    def _reorder(df: pd.DataFrame) -> pd.DataFrame:
        if df is None or len(df) == 0:
            return df
        front = [c for c in ["date", "symbol", "market_code", "name", "score", "reasons"] if c in df.columns]
        rest = [c for c in df.columns if c not in front]
        return df[front + rest]

    df_signals = _reorder(df_signals)
    df_shortlist = _reorder(df_shortlist)

    sig_path = save_signals_market(df_signals, base_dir, market)

    # append shortlist (history) instead of overwrite
    sh_path = save_shortlist_market_append(df_shortlist, base_dir, market)

    print("✔ signals:", sig_path, f"({len(df_signals)} lignes, date={df_signals['date'].max()})")
    print("✔ shortlist (history):", sh_path, f"(new rows={len(df_shortlist)}, seuil={score_threshold})")

    return df_signals, df_shortlist



In [ ]:
# === EXECUTION: Scoring + exports (per market) using results from previous FEATURES loop ===
# Preconditions:
#  - feat_by_market dict exists from the features loop: {market: df_feat_market}
#  - run_scoring_and_export(...) already defined

# === EXECUTION: Scoring + exports (per market) using results from previous FEATURES loop ===
# Preconditions:
#  - feat_by_market dict exists from the features loop: {market: df_feat_market}
#  - run_scoring_and_export(..., reprocess_mode=...) already defined

import pandas as pd
import os

BASE_CACHE_DIR = os.environ["BASE_CACHE_DIR"]

# NEW: choose processing mode
#  - "incremental"  : latest date only (current behavior when asof_date=None)
#  - "full"         : recompute ALL dates, overwrite parquet
#  - "full_append"  : recompute ALL dates, merge+dedup into parquet
REPROCESS_MODE = os.environ.get("REPROCESS_MODE", "incremental").strip().lower()
REPROCESS_MODE = "full_append"

signals_by_market = {}
shortlist_by_market = {}

for m, df_feat_m in feat_by_market.items():
    if df_feat_m is None or len(df_feat_m) == 0:
        print(f"⚠️ [{m}] Features vides, skip scoring.")
        continue

    signals_by_market[m], shortlist_by_market[m] = run_scoring_and_export(
        df_feat_market=df_feat_m,
        base_dir=BASE_CACHE_DIR,
        market=m,
        asof_date=None,         # None = latest date available (ignored in full/full_append)
        top_n=50,
        score_threshold=70,
        rvol_threshold=1.2,
        use_breakout="50",
        reprocess_mode=REPROCESS_MODE,   # NEW
    )

# Combine all shortlists (optional, multi-markets)
dfs = []
for m, df_sl_m in shortlist_by_market.items():
    if df_sl_m is not None and len(df_sl_m) > 0:
        dfs.append(df_sl_m.assign(market=m))

df_shortlist_all = (
    pd.concat(dfs, ignore_index=True)
    .sort_values(["score"], ascending=False)
    .reset_index(drop=True)
) if dfs else pd.DataFrame()

print("REPROCESS_MODE =", REPROCESS_MODE)
print("Shortlist combined:", len(df_shortlist_all))
df_shortlist_all.head(30)

✔ shortlist [eu] déjà ajoutée pour 2026-01-21 -> no-op
✔ signals: /content/drive/MyDrive/saxo/cache/signals_daily_eu.parquet (131484 lignes, dates=265)
✔ shortlist (history): /content/drive/MyDrive/saxo/cache/shortlist_eu.csv (new rows=12280)
✔ shortlist [us] déjà ajoutée pour 2026-01-21 -> no-op
✔ signals: /content/drive/MyDrive/saxo/cache/signals_daily_us.parquet (366291 lignes, dates=265)
✔ shortlist (history): /content/drive/MyDrive/saxo/cache/shortlist_us.csv (new rows=12907)
REPROCESS_MODE = full_append
Shortlist combined: 25187


,date,symbol,market_code,name,score,reasons,close,volume,mom_12_1,mom_pct,...,near52_scaled,trend_ok,breakout_ok,rvol_ok,rvol20,sma50,sma150,sma200,atr14,market
0,2026-01-20 00:00:00+00:00,CG,NASDAQ,The Carlyle Group Inc.,100,OK,61.8100,1716376.0,4.090931,0.992828,...,1.0,1,1,1,1.891034,19.994800,14.793400,13.451450,3.732500,us
1,2025-06-24 00:00:00+00:00,ENR,FSE,Siemens Energy AG,100,OK,91.1200,3021777.0,2.635174,0.996124,...,1.0,1,1,1,1.349121,77.639200,61.816067,55.074900,2.745714,eu
2,2025-07-17 00:00:00+00:00,RBLX,NYSE,Roblox Corporation,100,OK,122.1700,9826845.0,1.755435,0.986301,...,1.0,1,1,1,1.380522,93.942600,72.608267,66.264400,4.221614,us
3,2025-07-17 00:00:00+00:00,RKLB,NSC,Rocket Lab Corporation,100,OK,51.3300,46525984.0,7.040443,0.999315,...,1.0,1,1,1,1.616938,30.440600,25.613667,22.929325,2.859600,us
4,2025-07-18 00:00:00+00:00,MP,NYSE,MP Materials Corp.,100,OK,63.2200,28914800.0,2.355586,0.992466,...,1.0,1,1,1,1.320837,30.727000,25.369733,23.741200,5.142379,us
5,2025-07-18 00:00:00+00:00,HOOD,NASDAQ,Robinhood Markets Inc.,100,OK,109.7400,72954640.0,2.991366,0.996575,...,1.0,1,1,1,1.305205,77.223400,55.899133,49.454700,6.124143,us
6,2025-03-20 00:00:00+00:00,ZEG,LSE_SETS,Zegona Communications PLC,100,OK,685.0000,544859.0,1.922332,0.994129,...,1.0,1,1,1,1.343576,518.140000,404.593333,376.890000,26.071429,eu
7,2025-07-21 00:00:00+00:00,SFD,TSE,NXT Energy Solutions Inc.,100,OK,0.7500,100632.0,2.457265,0.993146,...,1.0,1,1,1,2.583998,0.566500,0.335867,0.306675,0.039286,us
8,2025-11-20 00:00:00+00:00,ECG,OOTC_NI,Eco-Growth Strategies Inc,100,OK,0.7498,745698.0,8.926190,0.998633,...,1.0,1,1,1,6.646955,0.284245,0.212041,0.198783,0.093621,us
9,2025-11-20 00:00:00+00:00,ABVX,NaN,NaN,100,OK,107.8000,201866.0,11.794227,0.999316,...,1.0,1,1,1,1.613914,80.626000,45.366000,35.565300,5.742857,us


## VIX

In [ ]:
import requests
import pandas as pd
from io import StringIO
import re

FGI_CSV_URL = "https://api.alternative.me/fng/?limit=0&format=csv"

def _extract_fgi_embedded_csv(text: str) -> str:
    """
    Extract the CSV-like block inside: "data": [ ... ]
    in the pseudo-JSON response. Returns CSV text.
    """
    # capture everything between '"data": [' and the matching closing ']' before '"metadata"'
    m = re.search(r'"data"\s*:\s*\[\s*(.*?)\s*\]\s*,\s*"metadata"', text, flags=re.S)
    if not m:
        raise ValueError("Cannot find embedded data block in FGI response.")
    block = m.group(1).strip()

    # remove possible leading/trailing commas and stray quotes
    block = block.strip().strip(",").strip().strip('"').strip()

    # keep only non-empty lines
    lines = [ln.strip() for ln in block.splitlines() if ln.strip()]
    if not lines:
        raise ValueError("Embedded data block is empty.")

    # Fix header if present but inverted vs data rows
    # Observed header: fng_value,fng_classification,date
    # Observed rows:   date,fng_value,fng_classification
    header = lines[0].replace(" ", "")
    if header.lower() == "fng_value,fng_classification,date":
        lines[0] = "date,fng_value,fng_classification"

    return "\n".join(lines)

def load_fgi_history_csv(url: str = FGI_CSV_URL) -> pd.DataFrame:
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    text = r.text.strip()

    # 1) If it looks like a proper CSV (starts with a header that contains commas)
    first_line = text.splitlines()[0].strip() if text else ""
    looks_like_csv = ("," in first_line) and (not first_line.startswith("{"))
    if looks_like_csv:
        df_raw = pd.read_csv(StringIO(text), sep=",", engine="python")
        return _normalize_fgi_df(df_raw)

    # 2) Try proper JSON (in case the provider fixes it later)
    if text.startswith("{") and '"data"' in text:
        try:
            j = r.json()
            data = j.get("data")
            if isinstance(data, list):
                df_raw = pd.DataFrame(data)
                return _normalize_fgi_df(df_raw)
            if isinstance(data, str):
                df_raw = pd.read_csv(StringIO(data), sep=",", engine="python")
                return _normalize_fgi_df(df_raw)
        except Exception:
            # fall through to pseudo-JSON parsing
            pass

        # 3) Pseudo-JSON (broken) => extract embedded CSV block
        csv_text = _extract_fgi_embedded_csv(text)
        df_raw = pd.read_csv(StringIO(csv_text), sep=",", engine="python")
        return _normalize_fgi_df(df_raw)

    raise ValueError("Unrecognized FGI response format.")

def _normalize_fgi_df(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize to columns: date (UTC normalized), value (numeric), classification (string)
    Supports both schemas:
      - timestamp,value,classification  (official alternative.me CSV schema)
      - fng_value,fng_classification,date  (your observed embedded schema)
      - date,value,classification (best-effort)
    """
    cols = {c.strip().lower(): c for c in df_raw.columns}

    # value
    if "value" in cols:
        value_col = cols["value"]
    elif "fng_value" in cols:
        value_col = cols["fng_value"]
    else:
        raise ValueError(f"Unexpected schema (no value): {df_raw.columns.tolist()}")

    # classification
    if "classification" in cols:
        class_col = cols["classification"]
    elif "fng_classification" in cols:
        class_col = cols["fng_classification"]
    else:
        class_col = None

    # date
    if "timestamp" in cols:
        df_raw["date"] = pd.to_datetime(df_raw[cols["timestamp"]], unit="s", utc=True, errors="coerce").dt.normalize()
    elif "date" in cols:
        # supports dd-mm-YYYY like 21-01-2026
        df_raw["date"] = pd.to_datetime(df_raw[cols["date"]], utc=True, errors="coerce", dayfirst=True).dt.normalize()
    else:
        raise ValueError(f"Unexpected schema (no timestamp/date): {df_raw.columns.tolist()}")

    df_raw["value"] = pd.to_numeric(df_raw[value_col], errors="coerce")
    df_raw["classification"] = df_raw[class_col].astype(str) if class_col else None

    df = (
        df_raw[["date", "value", "classification"]]
        .dropna(subset=["date", "value"])
        .sort_values("date")
        .reset_index(drop=True)
    )
    return df

def fgi_label(value: float) -> str:
    v = float(value)
    if v <= 25: return "Extreme Fear"
    if v <= 45: return "Fear"
    if v <= 55: return "Neutral"
    if v <= 75: return "Greed"
    return "Extreme Greed"

def add_fgi_to_tradeplan(
    df_plan: pd.DataFrame,
    base_dir: str,
    ma_window: int = 20,
    fgi_url: str = FGI_CSV_URL,
) -> pd.DataFrame:
    """
    Enrichit TOUTES les lignes du tradeplan avec l'état psychologique du marché (FGI).

    Colonnes ajoutées :
      - fgi_value        : valeur numérique actuelle
      - fgi_label        : Extreme Fear / Fear / Neutral / Greed / Extreme Greed
      - fgi_ma_window    : fenêtre MMA utilisée
      - fgi_ma           : valeur de la MMA du FGI
      - fgi_signal_ma    : ABOVE_MA / BELOW_MA / CROSS_UP / CROSS_DOWN / NO_MA_YET

    Le FGI est calculé UNE FOIS (dernier point dispo) et appliqué à toutes les lignes.
    """

    if df_plan is None or len(df_plan) == 0:
        return df_plan

    # 1) Load full FGI history (cleaned)
    df_fgi = load_fgi_history_csv(fgi_url)

    if df_fgi is None or len(df_fgi) < ma_window:
        # fallback: add empty columns
        out = df_plan.copy()
        out["fgi_value"] = None
        out["fgi_label"] = None
        out["fgi_ma_window"] = ma_window
        out["fgi_ma"] = None
        out["fgi_signal_ma"] = "NO_DATA"
        return out

    # 2) Compute FGI state
    d = df_fgi.copy().sort_values("date")
    d["fgi_ma"] = d["value"].rolling(int(ma_window)).mean()

    last = d.iloc[-1]
    prev = d.iloc[-2] if len(d) >= 2 else None

    fgi_value = float(last["value"])
    fgi_ma = float(last["fgi_ma"]) if pd.notna(last["fgi_ma"]) else None

    # Human readable label
    def _label(v: float) -> str:
        if v <= 25: return "Extreme Fear"
        if v <= 45: return "Fear"
        if v <= 55: return "Neutral"
        if v <= 75: return "Greed"
        return "Extreme Greed"

    fgi_label = _label(fgi_value)

    # MA signal
    if fgi_ma is None:
        sig = "NO_MA_YET"
    else:
        above = fgi_value >= fgi_ma
        if prev is None or pd.isna(prev["fgi_ma"]):
            sig = "ABOVE_MA" if above else "BELOW_MA"
        else:
            prev_above = float(prev["value"]) >= float(prev["fgi_ma"])
            if not prev_above and above:
                sig = "CROSS_UP"
            elif prev_above and not above:
                sig = "CROSS_DOWN"
            else:
                sig = "ABOVE_MA" if above else "BELOW_MA"

    # 3) Persist raw history for audit (optional, but useful)
    os.makedirs(base_dir, exist_ok=True)
    fgi_hist_path = os.path.join(base_dir, "fgi_history.csv")
    df_fgi.to_csv(fgi_hist_path, index=False)

    # 4) Attach to tradeplan (broadcast to all rows)
    out = df_plan.copy()
    out["fgi_value"] = fgi_value
    out["fgi_label"] = fgi_label
    out["fgi_ma_window"] = int(ma_window)
    out["fgi_ma"] = fgi_ma
    out["fgi_signal_ma"] = sig

    return out

## Trade Plan

À partir de shortlist_{market}.csv, tu ajoutes des colonnes actionnables :

	*	entry (par défaut : close du jour ou next_open si tu veux exécuter J+1)
	*	stop_initial (ex. close - 2*ATR14)
	*	risk_per_share = entry - stop
	*	position_size (si tu fixes un risque par trade, ex. 0,5% du capital)
	*	take-profit (optionnel) ou trailing (ATR)

In [ ]:
import os
import numpy as np
import pandas as pd

# --------- Trade plan (1% risk per trade) ---------

def build_trade_plan(
    df_shortlist: pd.DataFrame,
    capital: float,
    risk_pct: float = 0.01,          # baseline risk (applied when RISK_ON)
    atr_mult: float = 2.0,
    min_stop_pct: float = 0.05,
    max_positions: int | None = None,
    regime: str | None = None,       # "RISK_ON" | "RISK_SEMI_OFF" | "RISK_OFF"
) -> pd.DataFrame:
    """
    3-level regime support:
      - RISK_ON:       risk_pct as-is
      - RISK_SEMI_OFF: risk_pct * 0.5
      - RISK_OFF:      risk_pct * 0.0 (=> no valid positions)
    """
    df = df_shortlist.copy()
    df["symbol"] = df["symbol"].astype(str).str.upper()

    # Entry
    df["entry"] = pd.to_numeric(df["close"], errors="coerce")

    atr = pd.to_numeric(df.get("atr14"), errors="coerce")
    df["atr14"] = atr

    # Stop logic
    df["stop_atr"] = df["entry"] - atr_mult * df["atr14"]
    df["stop_minpct"] = df["entry"] * (1.0 - float(min_stop_pct))

    df["stop"] = df["stop_atr"]
    bad_atr = df["stop"].isna() | (df["stop"] >= df["entry"]) | ((df["entry"] - df["stop"]) / df["entry"] < (min_stop_pct / 2))
    df.loc[bad_atr, "stop"] = df.loc[bad_atr, "stop_minpct"]

    # Risk per share
    df["risk_per_share"] = df["entry"] - df["stop"]

    # --- Regime-based risk scaling ---
    regime_norm = (regime or "RISK_ON").strip().upper()
    risk_mult = {"RISK_ON": 1.0, "RISK_SEMI_OFF": 0.5, "RISK_OFF": 0.0}.get(regime_norm, 1.0)
    eff_risk_pct = float(risk_pct) * float(risk_mult)

    # Risk budget per trade
    risk_budget = float(capital) * float(eff_risk_pct)
    df["risk_budget"] = risk_budget

    # Shares (floor)
    if risk_budget <= 0:
        df["shares"] = pd.NA
    else:
        df["shares"] = np.floor(df["risk_budget"] / df["risk_per_share"]).astype("Int64")
        df.loc[df["shares"] < 1, "shares"] = pd.NA

    # Notional & checks
    df["notional"] = df["shares"].astype(float) * df["entry"]
    df["stop_pct"] = (df["risk_per_share"] / df["entry"]).replace([np.inf, -np.inf], np.nan)

    # Basic quality flags
    df["valid"] = (
        df["entry"].notna()
        & df["stop"].notna()
        & df["shares"].notna()
        & (df["risk_per_share"] > 0)
        & (df["shares"] >= 1)
    )

    # Attach regime info (useful in exports)
    df["regime"] = regime_norm
    df["risk_multiplier"] = risk_mult
    df["eff_risk_pct"] = eff_risk_pct

    # Sort
    sort_cols = ["score", "valid"] if "score" in df.columns else ["valid"]
    ascending = [False, False] if "score" in df.columns else [False]
    df = df.sort_values(sort_cols, ascending=ascending)

    if max_positions is not None:
        df = df.head(int(max_positions)).copy()

    cols = []
    for c in ["date", "symbol", "name","market_code","score", "reasons", "regime", "risk_multiplier", "eff_risk_pct"]:
        if c in df.columns:
            cols.append(c)
    cols += ["entry", "stop", "shares", "risk_budget", "risk_per_share", "stop_pct", "notional", "valid"]

    return df[cols].reset_index(drop=True)



def _latest_shortlist_only(df_shortlist: pd.DataFrame) -> pd.DataFrame:
    df = df_shortlist.copy()
    df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
    df["symbol"] = df["symbol"].astype(str).str.upper()
    df = df.dropna(subset=["date","symbol"])
    latest = df["date"].max()
    return df[df["date"] == latest].copy()

def tradeplan_path(base_dir: str, name: str = "tradeplan.csv") -> str:
    os.makedirs(base_dir, exist_ok=True)
    return os.path.join(base_dir, name)

def save_trade_plan_append(df_new: pd.DataFrame, out_path: str) -> str:
    """
    Append the new tradeplan at the TOP of the existing tradeplan file (history kept).
    Do nothing if this day's tradeplan has already been appended (prevents duplicates).

    Dedup key: (date, symbol). Idempotent across reruns for the same day.
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    if df_new is None or len(df_new) == 0:
        return out_path

    df_new = df_new.copy()
    if "date" not in df_new.columns:
        raise ValueError("df_new must contain a 'date' column for history append/dedup.")
    if "symbol" not in df_new.columns:
        raise ValueError("df_new must contain a 'symbol' column for history append/dedup.")

    df_new["date"] = pd.to_datetime(df_new["date"], utc=True, errors="coerce")
    df_new["symbol"] = df_new["symbol"].astype(str).str.upper()
    df_new = df_new.dropna(subset=["date","symbol"])

    latest_date = df_new["date"].max()
    if pd.isna(latest_date):
        return out_path

    # Load existing history if any
    if os.path.exists(out_path):
        df_old = pd.read_csv(out_path)
        if "date" in df_old.columns:
            df_old["date"] = pd.to_datetime(df_old["date"], utc=True, errors="coerce")
        if "symbol" in df_old.columns:
            df_old["symbol"] = df_old["symbol"].astype(str).str.upper()
    else:
        df_old = pd.DataFrame(columns=df_new.columns)

    # If this date already present with overlapping keys, no-op
    if len(df_old) > 0 and "date" in df_old.columns and (df_old["date"].max() == latest_date):
        old_keys = set(zip(df_old["date"], df_old["symbol"])) if {"date","symbol"}.issubset(df_old.columns) else set()
        new_keys = set(zip(df_new["date"], df_new["symbol"]))
        if len(old_keys.intersection(new_keys)) > 0:
            print(f"✔ tradeplan déjà ajouté pour {latest_date.date()} -> no-op")
            return out_path

    # Prepend + dedup on (date,symbol)
    df_combined = pd.concat([df_new, df_old], ignore_index=True)
    df_combined = df_combined.drop_duplicates(subset=["date","symbol"], keep="first")

    # Sort newest first (date desc, score desc if present)
    sort_cols = ["date"]
    ascending = [False]
    if "score" in df_combined.columns:
        sort_cols.append("score")
        ascending.append(False)

    df_combined = df_combined.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)
    df_combined.to_csv(out_path, index=False)

    print(f"✔ tradeplan append OK: {out_path} (added date {latest_date.date()})")
    return out_path


def compute_market_regime(df_index_ohlcv: pd.DataFrame) -> dict:
    """
    3-level market regime based on SMA50 and SMA200.

    Input: OHLCV for benchmark index with columns: date, close
    Output dict:
      {
        "date": <last_date>,
        "close": <float>,
        "sma50": <float>,
        "sma200": <float>,
        "regime": "RISK_ON" | "RISK_SEMI_OFF" | "RISK_OFF",
        "regime_ok": True/False,          # True only for RISK_ON
        "risk_multiplier": <float>,       # suggested risk scaling vs baseline
      }
    """
    d = df_index_ohlcv.copy()
    d["date"] = pd.to_datetime(d["date"], utc=True, errors="coerce")
    d["close"] = pd.to_numeric(d["close"], errors="coerce")
    d = d.dropna(subset=["date", "close"]).sort_values("date")

    d["sma50"] = d["close"].rolling(50).mean()
    d["sma200"] = d["close"].rolling(200).mean()

    last = d.iloc[-1]
    close = float(last["close"])
    sma50 = float(last["sma50"]) if pd.notna(last["sma50"]) else float("nan")
    sma200 = float(last["sma200"]) if pd.notna(last["sma200"]) else float("nan")

    # Default (if not enough history): be conservative
    if not (pd.notna(last["sma50"]) and pd.notna(last["sma200"])):
        regime = "RISK_OFF"
    else:
        if close > sma50 and sma50 > sma200:
            regime = "RISK_ON"
        elif close > sma200:
            regime = "RISK_SEMI_OFF"
        else:
            regime = "RISK_OFF"

    regime_ok = (regime == "RISK_ON")

    # Suggested risk scaling (you can tune later)
    risk_multiplier = {
        "RISK_ON": 1.0,
        "RISK_SEMI_OFF": 0.5,
        "RISK_OFF": 0.0,
    }[regime]

    return {
        "date": last["date"],
        "close": close,
        "sma50": sma50,
        "sma200": sma200,
        "regime": regime,
        "regime_ok": regime_ok,
        "risk_multiplier": risk_multiplier,
    }




In [ ]:
# === EXECUTION: Trade plan (1% risk) using results from previous SCORING loop ===
# Uses:
#  - shortlist_by_market (from scoring loop)
#  - build_trade_plan(...)
#  - save_trade_plan_append(...)  (append-on-top, no-dup)

import os
import pandas as pd

def _latest_shortlist_only(df_sl: pd.DataFrame) -> pd.DataFrame:
    if df_sl is None or len(df_sl) == 0:
        return pd.DataFrame(columns=df_sl.columns if df_sl is not None else [])
    d = df_sl.copy()
    d["date"] = pd.to_datetime(d["date"], utc=True, errors="coerce")
    d = d.dropna(subset=["date"])
    if len(d) == 0:
        return d
    latest = d["date"].max()
    return d[d["date"] == latest].copy()

BASE_CACHE_DIR = os.environ["BASE_CACHE_DIR"]
BASE_TRADE_DIR = os.environ["BASE_TRADE_DIR"]

capital = 100000  # <-- mets ton capital ici
risk_pct = 0.01   # 1%

tradeplan_by_market = {}

for m, df_sl_m in shortlist_by_market.items():
    if df_sl_m is None or len(df_sl_m) == 0:
        print(f"⚠️ [{m}] Shortlist vide, skip trade plan.")
        continue

    # Keep ONLY latest date
    df_sl_latest = _latest_shortlist_only(df_sl_m)

    if len(df_sl_latest) == 0:
        print(f"⚠️ [{m}] Aucun signal récent, skip.")
        continue

    tradeplan_by_market[m] = build_trade_plan(
        df_shortlist=df_sl_latest.assign(market=m),
        capital=capital,
        risk_pct=risk_pct,
        atr_mult=2.0,
        min_stop_pct=0.05,
        max_positions=None  # global cap applied after merge
    )

# --- Combine multi-markets ---
dfs = []
for m, df_tp_m in tradeplan_by_market.items():
    if df_tp_m is not None and len(df_tp_m) > 0:
        dfs.append(df_tp_m.assign(market=m))

df_plan_latest = (
    pd.concat(dfs, ignore_index=True)
    .sort_values(["valid", "score"], ascending=[False, False])
    .reset_index(drop=True)
) if dfs else pd.DataFrame()

# Global cap (e.g., max 20 new positions per day)
#MAX_POSITIONS_GLOBAL = 20
#df_plan_latest = df_plan_latest.head(MAX_POSITIONS_GLOBAL).copy()

# --- Inject Market Regime & reorder columns ---

regime_by_market = {}

for cfg in markets_cfg:
    m = cfg["market"]

    # benchmark OHLCV déjà téléchargé dans ton cache
    df_index_ohlcv = load_ohlcv_cache_market(BASE_CACHE_DIR, m)

    if df_index_ohlcv is None or len(df_index_ohlcv) < 220:
        raise RuntimeError(f"[{m}] Pas assez de données OHLCV pour calculer le regime")

    regime_by_market[m] = compute_market_regime_from_ohlcv(df_index_ohlcv)

    print(
        f"[{m}] Regime: "
        f"{'RISK_ON' if regime_by_market[m]['risk_on'] else 'RISK_OFF'} | "
        f"Close={regime_by_market[m]['close']:.2f} | "
        f"SMA200={regime_by_market[m]['sma200']:.2f}"
    )

df_plan_latest["regime"] = df_plan_latest["market"].map(
    lambda m: "RISK_ON" if regime_by_market[m]["risk_on"] else "RISK_OFF"
)

df_plan_latest["regime_ok"] = df_plan_latest["market"].map(
    lambda m: regime_by_market[m]["risk_on"]
)

# Reorder: score first
cols = df_plan_latest.columns.tolist()
cols = ["score"] + [c for c in cols if c != "score"]
df_plan_latest = df_plan_latest[cols]

# ---- VIX Data ----
df_plan_latest = add_fgi_to_tradeplan(df_plan_latest, BASE_CACHE_DIR, ma_window=20)

# Optionnel: mettre les colonnes FGI à gauche (après score si tu veux)
left_cols = ["score","date","symbol","name","market_code", "resaons","fgi_value","fgi_label","fgi_ma_window","fgi_ma","fgi_signal_ma"]
cols = [c for c in left_cols if c in df_plan_latest.columns] + [c for c in df_plan_latest.columns if c not in left_cols]
df_plan_latest = df_plan_latest[cols]

# --- Append to historical tradeplan (no overwrite, no duplicates) ---
out_path = os.path.join(BASE_TRADE_DIR, "tradeplan.csv")
save_trade_plan_append(df_plan_latest, out_path)

print("✔ tradeplan mis à jour (append historique):", out_path)
df_plan_latest.head(30)

[us] Regime: RISK_OFF | Close=122.00 | SMA200=340.58
[eu] Regime: RISK_OFF | Close=190.00 | SMA200=870.58


/tmp/ipython-input-2049841343.py:151: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined = pd.concat([df_new, df_old], ignore_index=True)


✔ tradeplan append OK: /content/drive/MyDrive/saxo/trades/tradeplan.csv (added date 2026-01-21)
✔ tradeplan mis à jour (append historique): /content/drive/MyDrive/saxo/trades/tradeplan.csv


,score,date,symbol,name,market_code,fgi_value,fgi_label,fgi_ma_window,fgi_ma,fgi_signal_ma,...,entry,stop,shares,risk_budget,risk_per_share,stop_pct,notional,valid,market,regime_ok
0,99,2026-01-21 00:00:00+00:00,INF,Informa Plc,LSE_SETS,24.0,Extreme Fear,20,35.65,BELOW_MA,...,909.40,783.642857,7,1000.0,125.757143,0.138286,6365.80,True,eu,False
1,90,2026-01-21 00:00:00+00:00,RKT,Reckitt Benckiser Group Plc,LSE_SETS,24.0,Extreme Fear,20,35.65,BELOW_MA,...,6140.00,5251.161429,1,1000.0,888.838571,0.144762,6140.00,True,eu,False
2,90,2026-01-21 00:00:00+00:00,RIO,Rio Tinto Plc,LSE_SETS,24.0,Extreme Fear,20,35.65,BELOW_MA,...,6408.00,5459.088000,1,1000.0,948.912000,0.148082,6408.00,True,eu,False
3,90,2026-01-21 00:00:00+00:00,IAG,International Consolidated Airlines Group SA,LSE_SETS,24.0,Extreme Fear,20,35.65,BELOW_MA,...,414.40,355.108857,16,1000.0,59.291143,0.143077,6630.40,True,eu,False
4,90,2026-01-21 00:00:00+00:00,DOC,Do & Co Restaurants & Catering AG,VIE,24.0,Extreme Fear,20,35.65,BELOW_MA,...,204.00,174.862571,34,1000.0,29.137429,0.142831,6936.00,True,us,False
5,89,2026-01-21 00:00:00+00:00,UTG,Unite Group,LSE_SETS,24.0,Extreme Fear,20,35.65,BELOW_MA,...,571.50,491.100000,12,1000.0,80.400000,0.140682,6858.00,True,eu,False
6,89,2026-01-21 00:00:00+00:00,EDEN,Edenred SA,PAR,24.0,Extreme Fear,20,35.65,BELOW_MA,...,17.30,14.703571,385,1000.0,2.596429,0.150083,6660.50,True,eu,False
7,89,2026-01-21 00:00:00+00:00,FRES,Fresnillo Plc,LSE_SETS,24.0,Extreme Fear,20,35.65,BELOW_MA,...,4060.00,3695.428571,2,1000.0,364.571429,0.089796,8120.00,True,eu,False
8,89,2026-01-21 00:00:00+00:00,SDR,Schroders Plc,LSE_SETS,24.0,Extreme Fear,20,35.65,BELOW_MA,...,449.20,384.670714,15,1000.0,64.529286,0.143654,6738.00,True,eu,False
9,89,2026-01-21 00:00:00+00:00,SRG,Snam,MIL,24.0,Extreme Fear,20,35.65,BELOW_MA,...,5.70,5.162857,1861,1000.0,0.537143,0.094236,10607.70,True,eu,False
